# 1. Data Cleaning & Auditing

This notebook section covers Stage 1 only: data discovery, cleaning, auditing, and before/after EDA.

## 1.1 Setup and Data Discovery
Load the 14 expected F1 tables and preserve raw copies for the audit.

In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
SENTINEL = "\\N"
INPUT_DIR = Path("/kaggle/input")
EVIDENCE_DIR = Path("/kaggle/working/stage1_cleaning_evidence")
EVIDENCE_DIR.mkdir(parents=True, exist_ok=True)

# A. Setup and data discovery
expected_tables = [
    "circuits", "constructors", "constructor_results", "constructor_standings",
    "drivers", "driver_standings", "races", "results", "qualifying",
    "lap_times", "pit_stops", "sprint_results", "seasons", "status"
]
all_csv_paths = sorted(INPUT_DIR.rglob("*.csv"))
found_by_stem = {}
for path in all_csv_paths:
    stem = path.stem.lower()
    if stem in expected_tables:
        found_by_stem.setdefault(stem, []).append(path)
missing_tables = [name for name in expected_tables if name not in found_by_stem]
ambiguous_tables = {name: paths for name, paths in found_by_stem.items() if len(paths) != 1}
if missing_tables or ambiguous_tables:
    raise FileNotFoundError(
        f"Expected exactly one CSV per table. Missing: {missing_tables}; "
        f"ambiguous: { {k: [str(p) for p in v] for k, v in ambiguous_tables.items()} }"
    )

csv_paths = {name: found_by_stem[name][0] for name in expected_tables}
input_folders = sorted({str(path.parent) for path in csv_paths.values()})
print("Discovered F1 dataset folder(s):")
for folder in input_folders:
    print(" -", folder)
print("\nDiscovered expected tables:", ", ".join(expected_tables))

# Read once, then keep raw_tables unchanged throughout this section.
raw_tables = {name: pd.read_csv(csv_paths[name]) for name in expected_tables}
raw_snapshots = {name: df.copy(deep=True) for name, df in raw_tables.items()}
clean_tables = {name: df.copy(deep=True) for name, df in raw_tables.items()}

primary_keys = {
    "circuits": ["circuitId"],
    "constructors": ["constructorId"],
    "constructor_results": ["constructorResultsId"],
    "constructor_standings": ["constructorStandingsId"],
    "drivers": ["driverId"],
    "driver_standings": ["driverStandingsId"],
    "races": ["raceId"],
    "results": ["resultId"],
    "qualifying": ["qualifyId"],
    "lap_times": ["raceId", "driverId", "lap"],
    "pit_stops": ["raceId", "driverId", "stop"],
    "sprint_results": ["resultId"],
    "seasons": ["year"],
    "status": ["statusId"]
}

def normalize_sentinels(frame):
    """Return a copy with only the literal Ergast sentinel represented as NA."""
    out = frame.copy(deep=True)
    for column in out.columns:
        series = out[column]
        if pd.api.types.is_object_dtype(series.dtype) or pd.api.types.is_string_dtype(series.dtype):
            out[column] = series.mask(series.eq(SENTINEL).fillna(False), pd.NA)
    return out

def count_literal_sentinels(frame):
    total = 0
    for column in frame.columns:
        series = frame[column]
        if pd.api.types.is_object_dtype(series.dtype) or pd.api.types.is_string_dtype(series.dtype):
            total += int(series.eq(SENTINEL).fillna(False).sum())
    return total

def key_audit_counts(frame, key_columns):
    key_values = normalize_sentinels(frame[key_columns])
    null_mask = key_values.isna().any(axis=1)
    duplicate_rows = int(key_values.loc[~null_mask].duplicated(keep=False).sum())
    return int(null_mask.sum()), duplicate_rows

race_years_numeric = pd.to_numeric(raw_tables["races"]["year"], errors="coerce").dropna()
race_year_min = int(race_years_numeric.min()) if len(race_years_numeric) else None
race_year_max = int(race_years_numeric.max()) if len(race_years_numeric) else None
print(f"\nRace-year range in attached races table: {race_year_min}–{race_year_max}")
if race_year_min is not None and (race_year_min < 1950 or race_year_max > 2024):
    print("REVIEW: the attached range is outside the expected 1950–2024 range; no year filtering was applied.")

initial_rows = []
initial_dtype_rows = []
for name, frame in raw_tables.items():
    null_keys, duplicate_keys = key_audit_counts(frame, primary_keys[name])
    initial_rows.append({
        "table": name,
        "rows": len(frame),
        "columns": len(frame.columns),
        "recognized_missing_values": int(frame.isna().sum().sum()),
        "literal_sentinel_values": count_literal_sentinels(frame),
        "exact_full_row_duplicate_rows": int(frame.duplicated().sum()),
        "primary_key": ", ".join(primary_keys[name]),
        "null_primary_key_rows": null_keys,
        "duplicate_primary_key_rows": duplicate_keys
    })
    for column, dtype in frame.dtypes.items():
        initial_dtype_rows.append({"table": name, "column": column, "dtype_before": str(dtype),
                                   "recognized_missing_values": int(frame[column].isna().sum()),
                                   "literal_sentinel_values": count_literal_sentinels(frame[[column]])})

table_overview = pd.DataFrame(initial_rows)
initial_dtypes = pd.DataFrame(initial_dtype_rows)

## 1.2 Initial EDA Before Cleaning
Summarize table shapes, missingness, sentinels, key issues, and duplicate groups before any cleaning.

In [ ]:


display(table_overview)
display(initial_dtypes)

# Race-year lookup for duplicate-group evidence.
races_for_year = normalize_sentinels(raw_tables["races"][["raceId", "year"]]).copy()
races_for_year["year_numeric"] = pd.to_numeric(races_for_year["year"], errors="coerce")
race_year_lookup = dict(zip(races_for_year["raceId"], races_for_year["year_numeric"]))

# Duplicate driver-race evidence is displayed before any resolution.
raw_results = raw_tables["results"]
raw_results_audit = normalize_sentinels(raw_results)
driver_race_keys = ["raceId", "driverId"]
valid_result_key = raw_results_audit[driver_race_keys].notna().all(axis=1)
raw_result_groups = raw_results_audit.loc[valid_result_key].groupby(driver_race_keys, sort=True).size()
duplicate_group_index = raw_result_groups[raw_result_groups > 1].index
duplicate_rows_mask = valid_result_key & raw_results_audit.duplicated(subset=driver_race_keys, keep=False)
duplicate_rows_before = raw_results_audit.loc[duplicate_rows_mask].copy()
duplicate_audit_rows = []
for (race_id, driver_id), group in duplicate_rows_before.groupby(driver_race_keys, sort=True):
    positions = pd.to_numeric(group.get("positionOrder"), errors="coerce").dropna()
    podium_labels = sorted(set(positions.le(3).astype(int).tolist()))
    duplicate_audit_rows.append({
        "raceId": race_id,
        "driverId": driver_id,
        "raceYear": race_year_lookup.get(race_id, pd.NA),
        "rows_in_group": len(group),
        "known_podium_labels": ", ".join("podium" if value else "not podium" for value in podium_labels),
        "conflicting_podium_labels": len(podium_labels) > 1
    })
duplicate_driver_race_audit = pd.DataFrame(duplicate_audit_rows, columns=[
    "raceId", "driverId", "raceYear", "rows_in_group",
    "known_podium_labels", "conflicting_podium_labels"
])

print(f"\nDuplicate (raceId, driverId) groups before resolution: {len(duplicate_group_index)}")
print(f"Rows participating in those groups: {len(duplicate_rows_before)}")
if len(duplicate_driver_race_audit):
    
    display(duplicate_driver_race_audit)
    detail_columns = [
        "raceId", "driverId", "resultId", "constructorId", "grid", "position",
        "positionText", "positionOrder", "points", "laps", "time", "milliseconds",
        "fastestLap", "fastestLapTime", "statusId"
    ]
    detail_columns = [column for column in detail_columns if column in duplicate_rows_before.columns]
    duplicate_rows_display = duplicate_rows_before[detail_columns].copy()
    duplicate_rows_display["raceYear"] = duplicate_rows_display["raceId"].map(race_year_lookup)
    display(duplicate_rows_display.sort_values(["raceYear", "raceId", "driverId"], kind="mergesort"))
    print("Years with duplicate groups:", sorted(pd.to_numeric(duplicate_driver_race_audit["raceYear"], errors="coerce").dropna().astype(int).unique().tolist()))
    print("Groups with conflicting known podium labels:", int(duplicate_driver_race_audit["conflicting_podium_labels"].sum()))
else:
    print("No duplicated driver-race groups were found.")

# Before-cleaning plots.
fig, ax = plt.subplots(figsize=(11, 7))
size_plot = table_overview.sort_values("rows", ascending=True)
ax.barh(size_plot["table"], size_plot["rows"], color="steelblue")
ax.set_title("F1 Table Sizes Before Cleaning")
ax.set_xlabel("Rows")
ax.set_ylabel("Table")
fig.tight_layout()
fig.savefig(EVIDENCE_DIR / "table_sizes_before.png", dpi=160, bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(15, 7))
missing_plot = table_overview.sort_values("recognized_missing_values", ascending=True)
axes[0].barh(missing_plot["table"], missing_plot["recognized_missing_values"], color="slateblue")
axes[0].set_title("Recognized Missing Values Before Cleaning")
axes[0].set_xlabel("Missing values")
axes[0].set_ylabel("Table")
sentinel_plot = table_overview.sort_values("literal_sentinel_values", ascending=True)
axes[1].barh(sentinel_plot["table"], sentinel_plot["literal_sentinel_values"], color="darkorange")
axes[1].set_title("Literal Sentinel Values Before Cleaning")
axes[1].set_xlabel("Literal sentinel values")
axes[1].set_ylabel("Table")
fig.tight_layout()
fig.savefig(EVIDENCE_DIR / "missingness_sentinels_before.png", dpi=160, bbox_inches="tight")
plt.show()

if len(duplicate_driver_race_audit):
    duplicate_year_counts = (
        duplicate_driver_race_audit.dropna(subset=["raceYear"])
        .groupby("raceYear").size().rename("duplicate_groups").reset_index()
    )
else:
    duplicate_year_counts = pd.DataFrame(columns=["raceYear", "duplicate_groups"])
display(duplicate_year_counts)
fig, ax = plt.subplots(figsize=(11, 5))
if len(duplicate_year_counts):
    sns.barplot(data=duplicate_year_counts, x="raceYear", y="duplicate_groups", color="firebrick", ax=ax)
    ax.tick_params(axis="x", rotation=45)
else:
    ax.text(0.5, 0.5, "No duplicate driver-race groups", ha="center", va="center", transform=ax.transAxes)
ax.set_title("Duplicated Driver-Race Groups by Year Before Resolution")
ax.set_xlabel("Race year")
ax.set_ylabel("Duplicate groups")
fig.tight_layout()
fig.savefig(EVIDENCE_DIR / "duplicate_driver_race_by_year.png", dpi=160, bbox_inches="tight")
plt.show()

## 1.3 Sentinel Cleaning
Replace literal sentinel strings with pandas missing values in clean_tables while retaining all rows and unchanged raw copies.

In [ ]:
# C–D. Sentinel cleaning and semantic typing, only on clean_tables.
for name, frame in list(clean_tables.items()):
    clean_tables[name] = normalize_sentinels(frame)
sentinels_after_replacement = {name: count_literal_sentinels(frame) for name, frame in clean_tables.items()}
print("\nLiteral sentinel values after replacement:", sum(sentinels_after_replacement.values()))

## 1.4 Semantic Typing
Convert selected dates and numeric values to usable types, parse durations into seconds, and retain conversion-failure reporting.

In [ ]:
conversion_rows = []
date_columns = []
for table_name, frame in clean_tables.items():
    for column in frame.columns:
        lower_name = column.lower()
        if lower_name in {"date", "dob"} or lower_name.endswith("_date"):
            original = frame[column].copy()
            converted = pd.to_datetime(original, errors="coerce")
            failures = int((original.notna() & converted.isna()).sum())
            conversion_rows.append({
                "table": table_name, "column": column, "converted_to": "datetime64[ns]",
                "non_missing_original_values": int(original.notna().sum()),
                "conversion_failures": failures
            })
            frame[column] = converted
            date_columns.append((table_name, column))

integer_columns = {
    "circuits": ["circuitId", "alt"],
    "constructors": ["constructorId"],
    "constructor_results": ["constructorResultsId", "raceId", "constructorId"],
    "constructor_standings": ["constructorStandingsId", "raceId", "constructorId", "position", "wins"],
    "drivers": ["driverId", "number"],
    "driver_standings": ["driverStandingsId", "raceId", "driverId", "position", "wins"],
    "races": ["raceId", "year", "round", "circuitId"],
    "results": ["resultId", "raceId", "driverId", "constructorId", "number", "grid", "position", "positionOrder", "laps", "statusId", "fastestLap", "rank"],
    "qualifying": ["qualifyId", "raceId", "driverId", "constructorId", "number", "position"],
    "lap_times": ["raceId", "driverId", "lap", "position", "milliseconds"],
    "pit_stops": ["raceId", "driverId", "stop", "lap", "milliseconds"],
    "sprint_results": ["resultId", "raceId", "driverId", "constructorId", "number", "grid", "position", "positionOrder", "laps", "statusId", "fastestLap", "rank"],
    "seasons": ["year"],
    "status": ["statusId"]
}
float_columns = {
    "circuits": ["lat", "lng"],
    "constructor_results": ["points"],
    "constructor_standings": ["points"],
    "driver_standings": ["points"],
    "results": ["points", "fastestLapSpeed"],
    "sprint_results": ["points", "fastestLapSpeed"]
}

def convert_numeric_column(frame, table_name, column, integer=False):
    original = frame[column].copy()
    numeric = pd.to_numeric(original, errors="coerce")
    failed = original.notna() & numeric.isna()
    non_finite = numeric.notna() & ~np.isfinite(numeric.astype("float64"))
    failed = failed | non_finite
    numeric = numeric.mask(non_finite, np.nan)
    if integer:
        non_integer = numeric.notna() & (np.mod(numeric.astype("float64"), 1) != 0)
        failed = failed | non_integer
        numeric = numeric.mask(non_integer, np.nan)
        frame[column] = pd.Series(pd.array(numeric, dtype="Int64"), index=frame.index, name=column)
        target_dtype = "Int64"
    else:
        frame[column] = pd.Series(pd.array(numeric, dtype="Float64"), index=frame.index, name=column)
        target_dtype = "Float64"
    conversion_rows.append({
        "table": table_name, "column": column, "converted_to": target_dtype,
        "non_missing_original_values": int(original.notna().sum()),
        "conversion_failures": int(failed.sum())
    })

for table_name, columns in integer_columns.items():
    frame = clean_tables[table_name]
    for column in columns:
        if column in frame.columns:
            convert_numeric_column(frame, table_name, column, integer=True)
for table_name, columns in float_columns.items():
    frame = clean_tables[table_name]
    for column in columns:
        if column in frame.columns:
            convert_numeric_column(frame, table_name, column, integer=False)


def parse_duration_seconds(value):
    if pd.isna(value):
        return np.nan

    text = str(value).strip()
    try:
        parts = text.split(":")
        if len(parts) == 1:
            return float(parts[0])
        if len(parts) == 2:
            minutes = float(parts[0])
            seconds = float(parts[1])
            return minutes * 60 + seconds
        if len(parts) == 3:
            hours = float(parts[0])
            minutes = float(parts[1])
            seconds = float(parts[2])
            return hours * 3600 + minutes * 60 + seconds
        return np.nan
    except (TypeError, ValueError):
        return np.nan


duration_test_cases = [
    ("1:23.456", 83.456),
    ("0:59.500", 59.5),
    ("25.713", 25.713),
    ("1:02:03.5", 3723.5),
]
assert np.isclose(parse_duration_seconds("1:23.456"), 83.456)
assert np.isclose(parse_duration_seconds("0:59.500"), 59.5)
assert np.isclose(parse_duration_seconds("25.713"), 25.713)
assert np.isclose(parse_duration_seconds("1:02:03.5"), 3723.5)
duration_test_results = pd.DataFrame([
    {"input_string": original, "expected_seconds": expected, "parsed_seconds": parse_duration_seconds(original)}
    for original, expected in duration_test_cases
])
display(duration_test_results)







duration_specs = [
    ("lap_times", "time", "time_seconds"),
    ("qualifying", "q1", "q1_seconds"),
    ("qualifying", "q2", "q2_seconds"),
    ("qualifying", "q3", "q3_seconds"),
    ("results", "fastestLapTime", "fastestLapTime_seconds"),
    ("sprint_results", "fastestLapTime", "fastestLapTime_seconds"),
    ("pit_stops", "duration", "duration_seconds")]

duration_sanity_rows = []
for table_name, source_column, _ in duration_specs:
    original_values = raw_tables[table_name][source_column].dropna().astype(str)
    original_values = original_values[
        original_values.str.strip().ne("") & original_values.ne(SENTINEL)
    ]
    if not original_values.empty:
        original_value = original_values.iloc[0]
        duration_sanity_rows.append({
            "table": table_name,
            "column": source_column,
            "original_duration": original_value,
            "converted_seconds": parse_duration_seconds(original_value),
        })
duration_sanity_table = pd.DataFrame(duration_sanity_rows)
display(duration_sanity_table)


duration_columns = []
missing_duration_sources = []
for table_name, source_column, output_column in duration_specs:
    frame = clean_tables[table_name]
    if source_column not in frame.columns:
        missing_duration_sources.append(f"{table_name}.{source_column}")
        continue
    original = frame[source_column].copy()
    converted = original.map(parse_duration_seconds)
    failures = int((original.notna() & converted.isna()).sum())
    frame[output_column] = pd.Series(pd.array(converted, dtype="Float64"), index=frame.index, name=output_column)
    conversion_rows.append({
        "table": table_name, "column": source_column, "converted_to": output_column + " (Float64 seconds)",
        "non_missing_original_values": int(original.notna().sum()),
        "conversion_failures": failures
    })
    duration_columns.append((table_name, output_column))

conversion_audit = pd.DataFrame(conversion_rows)
display(conversion_audit)
if missing_duration_sources:
    print("Duration source columns absent from this schema:", missing_duration_sources)
conversion_failures_total = int(conversion_audit["conversion_failures"].sum()) if len(conversion_audit) else 0
print("Total reported conversion failures:", conversion_failures_total)

## 1.5 Primary-Key Audit
Check the declared primary key in each table for missing or duplicate key values.

In [ ]:
# E. Primary-key audit on cleaned tables.
pk_rows = []
for table_name, key_columns in primary_keys.items():
    frame = clean_tables[table_name]
    null_key_rows, duplicate_key_rows = key_audit_counts(frame, key_columns)
    pk_rows.append({
        "table": table_name,
        "primary_key": ", ".join(key_columns),
        "null_key_rows": null_key_rows,
        "duplicate_key_rows": duplicate_key_rows,
        "passed": null_key_rows == 0 and duplicate_key_rows == 0
    })
pk_audit = pd.DataFrame(pk_rows)
display(pk_audit)

## 1.6 Foreign-Key Audit
Check each listed non-missing reference against its parent table and report unmatched rows.

In [ ]:
# F. Foreign-key audit. Null references are reported separately by their absence from non-null_fk_values.
foreign_keys = [
    ("races", "circuitId", "circuits", "circuitId"),
    ("races", "year", "seasons", "year"),
    ("results", "raceId", "races", "raceId"),
    ("results", "driverId", "drivers", "driverId"),
    ("results", "constructorId", "constructors", "constructorId"),
    ("results", "statusId", "status", "statusId"),
    ("qualifying", "raceId", "races", "raceId"),
    ("qualifying", "driverId", "drivers", "driverId"),
    ("qualifying", "constructorId", "constructors", "constructorId"),
    ("lap_times", "raceId", "races", "raceId"),
    ("lap_times", "driverId", "drivers", "driverId"),
    ("pit_stops", "raceId", "races", "raceId"),
    ("pit_stops", "driverId", "drivers", "driverId"),
    ("driver_standings", "raceId", "races", "raceId"),
    ("driver_standings", "driverId", "drivers", "driverId"),
    ("constructor_results", "raceId", "races", "raceId"),
    ("constructor_results", "constructorId", "constructors", "constructorId"),
    ("constructor_standings", "raceId", "races", "raceId"),
    ("constructor_standings", "constructorId", "constructors", "constructorId"),
    ("sprint_results", "raceId", "races", "raceId"),
    ("sprint_results", "driverId", "drivers", "driverId"),
    ("sprint_results", "constructorId", "constructors", "constructorId"),
    ("sprint_results", "statusId", "status", "statusId")
]
fk_rows = []
fk_violation_rows = []
for child_table, child_column, parent_table, parent_column in foreign_keys:
    child = clean_tables[child_table]
    parent = clean_tables[parent_table]
    non_null_values = child[child_column].dropna()
    parent_values = set(parent[parent_column].dropna().tolist())
    unmatched_values = set(non_null_values.unique().tolist()) - parent_values
    unmatched_mask = child[child_column].isin(unmatched_values)
    unmatched_count = int(unmatched_mask.sum())
    fk_rows.append({
        "child_table": child_table,
        "foreign_key": child_column,
        "parent_table": parent_table,
        "parent_key": parent_column,
        "non_null_fk_values": int(child[child_column].notna().sum()),
        "unmatched_rows": unmatched_count,
        "passed": unmatched_count == 0
    })
    if unmatched_count:
        bad_columns = [column for column in [child_column, "raceId", "driverId", "constructorId", "resultId"] if column in child.columns]
        bad = child.loc[unmatched_mask, bad_columns].copy().head(100)
        bad.insert(0, "child_table", child_table)
        bad["parent_table"] = parent_table
        bad["parent_key"] = parent_column
        fk_violation_rows.append(bad)
fk_audit = pd.DataFrame(fk_rows)
display(fk_audit)
if fk_violation_rows:
    fk_offending_rows = pd.concat(fk_violation_rows, ignore_index=True)
    print("Offending foreign-key rows (up to 100 per relationship):")
    display(fk_offending_rows)
else:
    fk_offending_rows = pd.DataFrame()
    print("No unmatched non-null foreign-key values were found.")

## 1.7 Driver-Race Duplicate Handling
Keep one result per race and driver using the approved grid and deterministic tie-break rule; audit the retained row and label conflicts.

In [ ]:
# G. One row per (raceId, driverId). Selection uses only pre-race grid plus a neutral ID tie-break.
clean_results = clean_tables["results"].copy(deep=True)
missing_driver_race_key_mask = clean_results[driver_race_keys].isna().any(axis=1)
results_without_driver_race_key = clean_results.loc[missing_driver_race_key_mask].copy()
graining_rows = clean_results.loc[~missing_driver_race_key_mask].copy()
graining_rows["_source_row_order"] = np.flatnonzero(~missing_driver_race_key_mask.to_numpy())
grid_numeric = pd.to_numeric(graining_rows["grid"], errors="coerce").astype("float64")
valid_grid = grid_numeric.notna() & grid_numeric.gt(0)
graining_rows["_grid_valid_rank"] = np.where(valid_grid, 0, 1)
graining_rows["_grid_position_sort"] = grid_numeric.where(valid_grid, np.inf)
result_id_numeric = pd.to_numeric(graining_rows["resultId"], errors="coerce").astype("float64")
graining_rows["_result_id_sort"] = result_id_numeric.fillna(np.inf)
results_driver_race = (
    graining_rows.sort_values(
        driver_race_keys + ["_grid_valid_rank", "_grid_position_sort", "_result_id_sort", "_source_row_order"],
        kind="mergesort"
    )
    .drop_duplicates(subset=driver_race_keys, keep="first")
    .drop(columns=["_source_row_order", "_grid_valid_rank", "_grid_position_sort", "_result_id_sort"])
    .reset_index(drop=True)
)

# The prediction row remains the grid-selected pre-race entry above. The
# driver-race outcome is separately aggregated across all valid result rows.
selected_predictor_rows_before_target = results_driver_race[["raceId", "driverId", "resultId", "constructorId", "grid"]].copy()
source_results_with_target = clean_results.loc[~missing_driver_race_key_mask].copy()
source_results_with_target["_position_order_numeric"] = pd.to_numeric(source_results_with_target["positionOrder"], errors="coerce")
source_results_with_target["_result_outcome_known"] = source_results_with_target["_position_order_numeric"].notna()
source_results_with_target["_result_podium"] = source_results_with_target["_position_order_numeric"].between(1, 3, inclusive="both")
driver_race_outcomes = (source_results_with_target.groupby(driver_race_keys, sort=False, as_index=False)
    .agg(driver_race_podium=("_result_podium", "any"),
         driver_race_outcome_known=("_result_outcome_known", "any")))
results_driver_race = results_driver_race.merge(driver_race_outcomes, on=driver_race_keys, how="left", validate="one_to_one")
results_driver_race["driver_race_podium"] = results_driver_race["driver_race_podium"].astype("int8")
results_driver_race["driver_race_outcome_known"] = results_driver_race["driver_race_outcome_known"].astype(bool)
results_driver_race["selected_entry_podium"] = pd.to_numeric(results_driver_race["positionOrder"], errors="coerce").between(1, 3, inclusive="both").astype("int8")
selected_predictor_rows_after_target = results_driver_race[["raceId", "driverId", "resultId", "constructorId", "grid"]]
selected_predictor_rows_preserved = selected_predictor_rows_before_target.equals(selected_predictor_rows_after_target)

duplicate_groups_resolved = int((results_driver_race.groupby(driver_race_keys).size() > 1).sum())
rows_removed_from_valid_grain = len(graining_rows) - len(results_driver_race)
rows_without_driver_race_key_count = len(results_without_driver_race_key)
duplicate_audit_rows = []
for (race_id, driver_id), group in duplicate_rows_before.groupby(driver_race_keys, sort=False):
    kept_row = results_driver_race.loc[
        results_driver_race["raceId"].eq(race_id) & results_driver_race["driverId"].eq(driver_id)
    ].iloc[0]
    if "positionOrder" in group.columns:
        known_positions = pd.to_numeric(group["positionOrder"], errors="coerce").dropna()
        known_labels = set(known_positions.map(lambda position: "podium" if position <= 3 else "not podium"))
    else:
        known_labels = set()
    duplicate_audit_rows.append({
        "raceId": race_id,
        "driverId": driver_id,
        "raceYear": race_year_lookup.get(race_id, pd.NA),
        "rows_in_group": len(group),
        "known_podium_labels": ", ".join(sorted(known_labels)) if known_labels else pd.NA,
        "conflicting_podium_labels": len(known_labels) > 1,
        "kept_resultId": kept_row.get("resultId", pd.NA),
        "kept_grid": kept_row.get("grid", pd.NA),
        "kept_constructorId": kept_row.get("constructorId", pd.NA),
        "kept_positionOrder": kept_row.get("positionOrder", pd.NA),
    })
duplicate_driver_race_audit = pd.DataFrame(duplicate_audit_rows)
duplicate_driver_race_audit["selected_row_podium"] = duplicate_driver_race_audit.apply(
    lambda row: int(results_driver_race.loc[
        results_driver_race["raceId"].eq(row["raceId"]) & results_driver_race["driverId"].eq(row["driverId"]),
        "selected_entry_podium"].iloc[0]), axis=1)
duplicate_driver_race_audit["driver_race_podium"] = duplicate_driver_race_audit.apply(
    lambda row: int(results_driver_race.loc[
        results_driver_race["raceId"].eq(row["raceId"]) & results_driver_race["driverId"].eq(row["driverId"]),
        "driver_race_podium"].iloc[0]), axis=1)
duplicate_driver_race_audit["podium_target_changed_by_aggregation"] = (
    duplicate_driver_race_audit["selected_row_podium"] != duplicate_driver_race_audit["driver_race_podium"])
display(duplicate_driver_race_audit)



print("\nDuplicate-resolution rule: keep the smallest valid positive grid position; treat grid=0, missing, or negative as unknown/worse; tie-break with the smallest resultId, then original row order.")
print("The selection uses no finish position, points, laps, race time, fastest lap, or status fields.")
print(f"Duplicate groups before: {len(duplicate_group_index)}")
print(f"Affected rows before: {len(duplicate_rows_before)}")
print(f"Rows removed from the valid modeling-grain copy by duplicate resolution: {rows_removed_from_valid_grain}")
print(f"Rows excluded from that copy because raceId or driverId is missing: {rows_without_driver_race_key_count}")
print(f"Unique driver-race rows after resolution: {len(results_driver_race)}")
print(f"Duplicate rows remaining in results_driver_race: {duplicate_groups_resolved}")
print(f"Duplicate groups with conflicting known podium labels: {int(duplicate_driver_race_audit['conflicting_podium_labels'].sum()) if len(duplicate_driver_race_audit) else 0}")
if len(duplicate_driver_race_audit):
        
        
    display(duplicate_driver_race_audit)

## 1.8 Post-Cleaning EDA
Compare missingness on the same original columns before and after cleaning; report derived-column missingness separately and regenerate evidence. Replacing literal sentinels with pandas missing values makes previously hidden missingness visible to pandas; it does not create missing information or remove rows.

In [ ]:
# H. Post-cleaning EDA and before/after summaries.
after_rows = []
clean_dtype_rows = []
for name, frame in clean_tables.items():
    original_columns = raw_tables[name].columns
    derived_columns = [column for column in frame.columns if column not in original_columns]
    after_rows.append({
        "table": name,
        "rows_after": len(frame),
        "columns_after": len(frame.columns),
        "recognized_missing_values_after": int(frame[original_columns].isna().sum().sum()),
        "derived_missing_values_after": int(frame[derived_columns].isna().sum().sum()),
        "literal_sentinel_values_after": count_literal_sentinels(frame[original_columns]),
        "exact_full_row_duplicate_rows_after": int(frame.duplicated().sum())
    })
    for column, dtype in frame.dtypes.items():
        clean_dtype_rows.append({"table": name, "column": column, "dtype_after": str(dtype)})
after_summary = pd.DataFrame(after_rows)
before_after_summary = table_overview[[
    "table", "rows", "columns", "recognized_missing_values", "literal_sentinel_values",
    "exact_full_row_duplicate_rows"
]].merge(after_summary, on="table", how="left")
before_after_summary = before_after_summary.rename(columns={
    "rows": "rows_before", "columns": "columns_before",
    "recognized_missing_values": "recognized_missing_values_before",
    "literal_sentinel_values": "literal_sentinel_values_before",
    "exact_full_row_duplicate_rows": "exact_full_row_duplicate_rows_before"
})
clean_dtypes = pd.DataFrame(clean_dtype_rows)
display(before_after_summary)
display(clean_dtypes)

fig, axes = plt.subplots(2, 2, figsize=(16, 13))
plot_names = before_after_summary["table"].tolist()
y = np.arange(len(plot_names))
bar_h = 0.38
axes[0, 0].barh(y - bar_h / 2, before_after_summary["rows_before"], height=bar_h, label="Before", color="gray")
axes[0, 0].barh(y + bar_h / 2, before_after_summary["rows_after"], height=bar_h, label="After", color="seagreen")
axes[0, 0].set_yticks(y, plot_names)
axes[0, 0].set_title("Rows Before vs After")
axes[0, 0].set_xlabel("Rows")
axes[0, 0].set_ylabel("Table")
axes[0, 0].legend()
axes[0, 1].barh(y - bar_h / 2, before_after_summary["recognized_missing_values_before"], height=bar_h, label="Before", color="gray")
axes[0, 1].barh(y + bar_h / 2, before_after_summary["recognized_missing_values_after"], height=bar_h, label="After", color="royalblue")
axes[0, 1].set_yticks(y, plot_names)
axes[0, 1].set_title("Recognized Missing Values Before vs After")
axes[0, 1].set_xlabel("Missing values")
axes[0, 1].set_ylabel("Table")
axes[0, 1].legend()
axes[1, 0].barh(y - bar_h / 2, before_after_summary["literal_sentinel_values_before"], height=bar_h, label="Before", color="gray")
axes[1, 0].barh(y + bar_h / 2, before_after_summary["literal_sentinel_values_after"], height=bar_h, label="After", color="darkorange")
axes[1, 0].set_yticks(y, plot_names)
axes[1, 0].set_title("Literal Sentinels Before vs After")
axes[1, 0].set_xlabel("Literal sentinel values")
axes[1, 0].set_ylabel("Table")
axes[1, 0].legend()
axes[1, 1].bar([0, 1], [len(duplicate_group_index), duplicate_groups_resolved], color=["firebrick", "seagreen"])
axes[1, 1].set_xticks([0, 1], ["Before resolution", "After resolution"])
axes[1, 1].set_title("Duplicated Driver-Race Rows")
axes[1, 1].set_ylabel("Duplicate driver-race groups")
fig.tight_layout()
fig.savefig(EVIDENCE_DIR / "cleaning_before_after.png", dpi=160, bbox_inches="tight")
plt.show()

# Save compact evidence tables alongside the displayed notebook results.
table_overview.to_csv(EVIDENCE_DIR / "table_overview.csv", index=False)
pk_audit.to_csv(EVIDENCE_DIR / "pk_audit.csv", index=False)
fk_audit.to_csv(EVIDENCE_DIR / "fk_audit.csv", index=False)
duplicate_driver_race_audit.to_csv(EVIDENCE_DIR / "duplicate_driver_race_audit.csv", index=False)
duplicate_rows_display.to_csv(EVIDENCE_DIR / "duplicate_driver_race_rows.csv", index=False) if len(duplicate_rows_before) else pd.DataFrame().to_csv(EVIDENCE_DIR / "duplicate_driver_race_rows.csv", index=False)
before_after_summary.to_csv(EVIDENCE_DIR / "before_after_cleaning_summary.csv", index=False)
conversion_audit.to_csv(EVIDENCE_DIR / "conversion_audit.csv", index=False)

## 1.9 Final Validation
Confirm source immutability, sentinel and type checks, all key audits, and the unique race-driver grain.

In [ ]:
# I. Final validation. Genuine PK/FK exceptions remain visible in their audit tables.
raw_tables_unchanged = all(raw_tables[name].equals(raw_snapshots[name]) for name in expected_tables)
literal_sentinels_after_total = sum(count_literal_sentinels(frame) for frame in clean_tables.values())
date_dtypes_ok = all(pd.api.types.is_datetime64_any_dtype(clean_tables[table][column]) for table, column in date_columns)
duration_dtypes_ok = all(pd.api.types.is_numeric_dtype(clean_tables[table][column]) for table, column in duration_columns)
pk_audit_complete = len(pk_audit) == len(expected_tables)
fk_audit_complete = len(fk_audit) == len(foreign_keys)
pk_audit_success = bool(pk_audit['passed'].all())
fk_audit_success = bool((fk_audit['unmatched_rows'] == 0).all())
expected_duplicate_years = [1950, 1951, 1952, 1953, 1954, 1955, 1956, 1957, 1958, 1960, 1961, 1962, 1964, 1978]
duplicate_years_after = sorted(pd.to_numeric(duplicate_driver_race_audit['raceYear'], errors='coerce').dropna().astype(int).unique().tolist())
duplicate_race_year_missing = int(duplicate_driver_race_audit.loc[duplicate_driver_race_audit['raceId'].notna(), 'raceYear'].isna().sum())
duplicate_race_years_valid = duplicate_race_year_missing == 0 and duplicate_years_after == expected_duplicate_years
results_driver_race_unique = not results_driver_race.duplicated(subset=driver_race_keys).any()
driver_race_outcome_reference = (source_results_with_target.groupby(driver_race_keys, sort=False)
    .agg(expected_podium=("_result_podium", "any"), expected_known=("_result_outcome_known", "any")).reset_index())
driver_race_outcome_check = results_driver_race[["raceId", "driverId", "driver_race_podium", "driver_race_outcome_known"]].merge(
    driver_race_outcome_reference, on=driver_race_keys, how="left", validate="one_to_one")
driver_race_labels_match_all_source_rows = (
    driver_race_outcome_check["driver_race_podium"].eq(driver_race_outcome_check["expected_podium"].astype("int8")).all()
    and driver_race_outcome_check["driver_race_outcome_known"].eq(driver_race_outcome_check["expected_known"]).all())
driver_race_target_changed_groups = int(duplicate_driver_race_audit["podium_target_changed_by_aggregation"].sum())
driver_race_target_changes_valid = driver_race_target_changed_groups == 17

assert raw_tables_unchanged, 'Raw input DataFrames changed during cleaning.'
assert literal_sentinels_after_total == 0, 'Literal sentinel values remain in clean_tables.'
assert date_dtypes_ok, 'At least one converted date field is not datetime typed.'
assert duration_dtypes_ok, 'At least one converted duration column is not numeric.'
assert pk_audit_complete, 'Primary-key audit is incomplete.'
assert fk_audit_complete, 'Foreign-key audit is incomplete.'
assert pk_audit['passed'].all(), 'At least one primary-key constraint failed.'
assert (fk_audit['unmatched_rows'] == 0).all(), 'At least one foreign-key relationship has unmatched rows.'
assert duplicate_race_year_missing == 0, 'Duplicate audit has unresolved race years.'
assert duplicate_years_after == expected_duplicate_years, 'Duplicate race years do not match the established year list.'
assert results_driver_race_unique, 'results_driver_race is not unique on (raceId, driverId).'
assert driver_race_labels_match_all_source_rows, 'Driver-race podium target does not match the any-result-row rule.'
assert driver_race_target_changes_valid, 'Expected 17 historical driver-race targets to change after aggregation.'
assert selected_predictor_rows_preserved, 'Outcome aggregation changed the pre-race selected predictor rows.'

validation_summary = pd.DataFrame([
    {'check': 'Raw input DataFrames unchanged', 'passed': raw_tables_unchanged},
    {'check': 'Zero literal sentinels in clean_tables', 'passed': literal_sentinels_after_total == 0},
    {'check': 'Date columns have datetime dtype', 'passed': date_dtypes_ok},
    {'check': 'Duration columns are numeric', 'passed': duration_dtypes_ok},
    {'check': 'PK audit completed for all tables', 'passed': pk_audit_complete},
    {'check': 'FK audit completed for all listed relationships', 'passed': fk_audit_complete},
    {'check': 'All primary-key constraints passed', 'passed': pk_audit_success},
    {'check': 'All audited foreign keys have zero unmatched rows', 'passed': fk_audit_success},
    {'check': 'Duplicate race years are complete and expected', 'passed': duplicate_race_years_valid},
    {'check': 'Unique (raceId, driverId) modeling grain', 'passed': results_driver_race_unique},
    {'check': 'Driver-race podium equals any source result podium', 'passed': driver_race_labels_match_all_source_rows},
    {'check': '17 historical duplicate targets corrected', 'passed': driver_race_target_changes_valid},
    {'check': 'Grid-selected predictor rows preserved', 'passed': selected_predictor_rows_preserved},
])
display(validation_summary)
print('PK passed count:', int(pk_audit['passed'].sum()), '/', len(pk_audit))
print('FK unmatched count:', int(fk_audit['unmatched_rows'].sum()))
print('Duplicate groups:', len(duplicate_driver_race_audit))
print('Corrected duplicate years:', duplicate_years_after)
print('Missing raceYear values in final duplicate audit:', duplicate_race_year_missing)
validation_summary.to_csv(EVIDENCE_DIR / 'stage1_validation_summary.csv', index=False)
print('Driver-race target changes from source-row aggregation:', driver_race_target_changed_groups)
duplicate_driver_race_audit.to_csv(EVIDENCE_DIR / 'duplicate_driver_race_target_audit.csv', index=False)


# 2. Data-Engineering Questions

## 2.1 Front-row start to podium by circuit

**Grain and preparation:** one unique driver-race row, joined to one race and circuit with validated many-to-one merges. A front-row start means actual starting grid 1 or 2; qualifying position is audited separately because penalties can change the grid. Podium is an outcome (positionOrder 1-3); positionOrder 0 is treated as unclassified, not a podium.

In [ ]:
# 2.1 Front-row start to podium by circuit
# Grain: one cleaned driver-race row from results_driver_race, joined to one race and one circuit.
from pathlib import Path
from IPython.display import display, Markdown

STAGE2_EVIDENCE_DIR = Path("/kaggle/working/stage2_data_engineering_evidence")
STAGE2_EVIDENCE_DIR.mkdir(parents=True, exist_ok=True)

# Validated many-to-one joins preserve the driver-race grain.
q1_races = clean_tables["races"][["raceId", "year", "name", "circuitId"]].rename(columns={"name": "race_name"})
q1_circuits = clean_tables["circuits"][["circuitId", "name", "country"]].rename(columns={"name": "circuit_name", "country": "circuit_country"})
q1_base = results_driver_race[["raceId", "driverId", "grid", "positionOrder", "driver_race_podium", "driver_race_outcome_known"]].copy()
q1_base = q1_base.merge(q1_races, on="raceId", how="left", validate="many_to_one")
q1_base = q1_base.merge(q1_circuits, on="circuitId", how="left", validate="many_to_one")
q1_base["year"] = pd.to_numeric(q1_base["year"], errors="coerce")
q1_base["grid"] = pd.to_numeric(q1_base["grid"], errors="coerce")
q1_base["positionOrder"] = pd.to_numeric(q1_base["positionOrder"], errors="coerce")
# In Ergast data, positionOrder=0 denotes an unclassified result, not a podium.
q1_base["podium"] = q1_base["driver_race_podium"].astype(bool)

q1_missingness = pd.DataFrame({"field": ["year", "circuitId", "circuit_name", "grid", "positionOrder"],
                                "missing_rows": [int(q1_base[c].isna().sum()) for c in ["year", "circuitId", "circuit_name", "grid", "positionOrder"]]})
q1_front = q1_base.loc[q1_base["grid"].isin([1, 2])].copy()
q1_front["era"] = np.where(q1_front["year"].between(1950, 2013), "Pre-hybrid (1950-2013)",
                            np.where(q1_front["year"].between(2014, 2024), "Hybrid (2014-2024)", pd.NA))
q1_front = q1_front.dropna(subset=["era", "circuitId", "circuit_name"])
print("Q1 grain: one unique driver-race row; tables joined: results_driver_race + races + circuits.")
print("Missingness before filters:")
display(q1_missingness)
print(f"Driver-race rows before front-row filter: {len(q1_base):,}; front-row starts used: {len(q1_front):,}; circuits represented: {q1_front['circuitId'].nunique():,}; years: {int(q1_front['year'].min())}-{int(q1_front['year'].max())}.")
print(f"Rows not analyzed as front-row starts (grid missing, <=0, or >2): {len(q1_base) - int(q1_base['grid'].isin([1, 2]).sum()):,}.")

def q1_circuit_metrics(frame):
    out = (frame.groupby(["circuitId", "circuit_name", "circuit_country"], dropna=False)
           .agg(front_row_starts=("podium", "size"), podiums_from_front_row=("podium", "sum"))
           .reset_index())
    out["conversion_rate"] = out["podiums_from_front_row"] / out["front_row_starts"]
    out = out.sort_values(["conversion_rate", "front_row_starts", "circuit_name"],
                          ascending=[False, False, True], kind="mergesort").reset_index(drop=True)
    out["rank"] = np.arange(1, len(out) + 1)
    return out

q1_all_time = q1_circuit_metrics(q1_front)
q1_era_tables = []
for era_name, year_range in [("Pre-hybrid (1950-2013)", (1950, 2013)), ("Hybrid (2014-2024)", (2014, 2024))]:
    era_metrics = q1_circuit_metrics(q1_front.loc[q1_front["year"].between(*year_range)]).copy()
    era_metrics.insert(0, "era", era_name)
    q1_era_tables.append(era_metrics)
q1_by_era = pd.concat(q1_era_tables, ignore_index=True)
q1_pre = q1_by_era.query("era == 'Pre-hybrid (1950-2013)'").rename(columns={
    "conversion_rate": "pre_hybrid_rate", "front_row_starts": "pre_hybrid_starts", "podiums_from_front_row": "pre_hybrid_podiums", "rank": "pre_hybrid_rank"})
q1_hyb = q1_by_era.query("era == 'Hybrid (2014-2024)'").rename(columns={
    "conversion_rate": "hybrid_rate", "front_row_starts": "hybrid_starts", "podiums_from_front_row": "hybrid_podiums", "rank": "hybrid_rank"})
q1_era_comparison = q1_pre[["circuitId", "circuit_name", "pre_hybrid_rate", "pre_hybrid_starts", "pre_hybrid_rank"]].merge(
    q1_hyb[["circuitId", "hybrid_rate", "hybrid_starts", "hybrid_rank"]], on="circuitId", how="inner", validate="one_to_one")
q1_era_comparison["rank_change_hybrid_minus_pre"] = q1_era_comparison["hybrid_rank"] - q1_era_comparison["pre_hybrid_rank"]
q1_era_comparison["rate_change_percentage_points"] = 100 * (q1_era_comparison["hybrid_rate"] - q1_era_comparison["pre_hybrid_rate"])
q1_era_comparison = q1_era_comparison.sort_values(["hybrid_rank", "pre_hybrid_rank", "circuit_name"], kind="mergesort")
print("All-time circuit ranking (complete table; denominators retained):")
display(q1_all_time)
print("Top circuits by era (complete table saved; top 5 shown here):")
display(q1_by_era.groupby("era", sort=False, group_keys=False).head(5))
print("Circuits represented in both eras:")
display(q1_era_comparison)

# Qualifying position is audited separately and is not used to define a front-row start.
q1_qual_raw = clean_tables["qualifying"][["raceId", "driverId", "qualifyId", "position"]].copy()
q1_qual_dupe_mask = q1_qual_raw.duplicated(["raceId", "driverId"], keep=False)
q1_qual_duplicate_rows = int(q1_qual_dupe_mask.sum())
q1_qual_duplicate_groups = int(q1_qual_raw.loc[q1_qual_dupe_mask].groupby(["raceId", "driverId"]).ngroups)
q1_qual = q1_qual_raw.dropna(subset=["raceId", "driverId"]).sort_values(["raceId", "driverId", "qualifyId"], kind="mergesort").drop_duplicates(["raceId", "driverId"], keep="first")
q1_qual = q1_qual.rename(columns={"position": "qualifying_position"})
q1_qual["qualifying_position"] = pd.to_numeric(q1_qual["qualifying_position"], errors="coerce")
q1_qual_audit = q1_base.merge(q1_qual[["raceId", "driverId", "qualifying_position"]], on=["raceId", "driverId"], how="left", validate="one_to_one", indicator=True)
q1_qual_audit["grid"] = pd.to_numeric(q1_qual_audit["grid"], errors="coerce")
q1_both = q1_qual_audit.dropna(subset=["qualifying_position", "grid"]).copy()
q1_both["qual_position_equals_grid"] = q1_both["qualifying_position"].eq(q1_both["grid"])
q1_mismatch_count = int((~q1_both["qual_position_equals_grid"]).sum())
q1_mismatch_pct = 100 * q1_mismatch_count / len(q1_both) if len(q1_both) else np.nan
q1_diff_examples = q1_both.loc[~q1_both["qual_position_equals_grid"], ["year", "race_name", "driverId", "qualifying_position", "grid"]].sort_values(["year", "race_name", "driverId"], kind="mergesort").head(10)
print(f"Qualifying (raceId, driverId) duplicate rows: {q1_qual_duplicate_rows:,}; duplicate groups: {q1_qual_duplicate_groups:,}. Deterministic audit representative: smallest qualifyId.")
print(f"Matched driver-race rows with both numeric qualifying position and grid: {len(q1_both):,}; equal: {int(q1_both['qual_position_equals_grid'].sum()):,}; different: {q1_mismatch_count:,} ({q1_mismatch_pct:.2f}%).")
print("Examples where qualifying position differs from the actual starting grid:")
display(q1_diff_examples)
q1_qual_audit.to_csv(STAGE2_EVIDENCE_DIR / "q1_qualifying_vs_grid_audit.csv", index=False)
q1_missingness.to_csv(STAGE2_EVIDENCE_DIR / "q1_missingness_audit.csv", index=False)
q1_all_time.to_csv(STAGE2_EVIDENCE_DIR / "q1_front_row_all_time.csv", index=False)
q1_by_era.to_csv(STAGE2_EVIDENCE_DIR / "q1_front_row_by_era.csv", index=False)
q1_era_comparison.to_csv(STAGE2_EVIDENCE_DIR / "q1_era_comparison.csv", index=False)
pd.DataFrame([{"matched_rows_with_both": len(q1_both), "qualifying_equals_grid": int(q1_both["qual_position_equals_grid"].sum()), "qualifying_differs_from_grid": q1_mismatch_count, "mismatch_percent": q1_mismatch_pct, "duplicate_groups_in_qualifying": q1_qual_duplicate_groups}]).to_csv(STAGE2_EVIDENCE_DIR / "q1_qualifying_vs_grid_summary.csv", index=False)

# Plot 1: leading all-time circuits, with start counts labelled.
q1_plot = q1_all_time.head(10).sort_values("conversion_rate")
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(q1_plot["circuit_name"], q1_plot["conversion_rate"], color="steelblue")
for y, (_, row) in enumerate(q1_plot.iterrows()): ax.text(row["conversion_rate"] + 0.01, y, f"{row['podiums_from_front_row']}/{row['front_row_starts']} starts", va="center", fontsize=8)
ax.set_xlim(0, min(1.25, max(1.05, float(q1_plot["conversion_rate"].max()) + 0.25)))
ax.xaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0))
ax.set_title("Top 10 Circuits: Podiums from Front-Row Starts (All Time)")
ax.set_xlabel("Podium conversion rate"); ax.set_ylabel("Circuit")
fig.tight_layout(); fig.savefig(STAGE2_EVIDENCE_DIR / "q1_all_time_top_circuits.png", dpi=160, bbox_inches="tight"); plt.show()

# Plot 2: top 10 per era; full deterministic era rankings remain in the CSV.
fig, axes = plt.subplots(1, 2, figsize=(16, 8))
for ax, era_name in zip(axes, ["Pre-hybrid (1950-2013)", "Hybrid (2014-2024)"]):
    part = q1_by_era.loc[q1_by_era["era"].eq(era_name)].head(10).sort_values("conversion_rate")
    ax.barh(part["circuit_name"], part["conversion_rate"], color="darkorange" if "Pre" in era_name else "seagreen")
    for y, (_, row) in enumerate(part.iterrows()): ax.text(row["conversion_rate"] + 0.01, y, f"n={row['front_row_starts']}", va="center", fontsize=8)
    ax.set_xlim(0, min(1.25, max(1.05, float(part["conversion_rate"].max()) + 0.2)))
    ax.xaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0)); ax.set_title(era_name); ax.set_xlabel("Podium conversion rate"); ax.set_ylabel("Circuit")
fig.suptitle("Leading Circuit Conversion Rates by Era (Top 10; Complete Tables in CSV)"); fig.tight_layout(); fig.savefig(STAGE2_EVIDENCE_DIR / "q1_top_circuits_by_era.png", dpi=160, bbox_inches="tight"); plt.show()

q1_top_names = lambda frame: ", ".join(f"{r.circuit_name} ({r.conversion_rate:.1%}, n={r.front_row_starts})" for r in frame.head(3).itertuples())
print("Interpretation (descriptive, not causal):")
q1_pre_top_text = q1_top_names(q1_by_era.loc[q1_by_era["era"].eq("Pre-hybrid (1950-2013)")])
q1_hybrid_top_text = q1_top_names(q1_by_era.loc[q1_by_era["era"].eq("Hybrid (2014-2024)")])
q1_all_top_text = q1_top_names(q1_all_time)
q1_rank_change_text = "differ" if (not q1_era_comparison.empty and q1_era_comparison["rank_change_hybrid_minus_pre"].ne(0).any()) else "are broadly similar"
display(Markdown("All-time leaders: " + q1_all_top_text + ". Pre-hybrid leaders: " + q1_pre_top_text + ". Hybrid-era leaders: " + q1_hybrid_top_text + ". The leading rates/ranks " + q1_rank_change_text + " between eras. Interpret high percentages with their denominators because some circuits have few starts. Actual grid, not qualifying position, defines the front row because penalties can change the starting order; the matched qualifying audit finds a " + f"{q1_mismatch_pct:.2f}%" + " mismatch rate. Historical coverage and circuit sample sizes vary."))
print("Q1 evidence saved under:", STAGE2_EVIDENCE_DIR)

## 2.2 Home-country podium comparison controlling for grid

**Grain and joins:** one driver-race row, with validated joins to race year, circuit country, and driver nationality. Nationality is a dataset proxy for home country; exact positive grid positions are controlled by direct standardization. Unknown country mappings are excluded from home/away comparisons and reported.

In [ ]:
# 2.2 Home-country podium comparison controlling for starting grid
# Nationality is mapped explicitly to a country; unrecognized/missing rows are never called Away.
q2_races = clean_tables["races"][["raceId", "year", "circuitId"]].copy()
q2_circuits = clean_tables["circuits"][["circuitId", "name", "country"]].rename(columns={"name": "circuit_name", "country": "circuit_country"})
q2_drivers = clean_tables["drivers"][["driverId", "nationality"]].copy()
q2_base = results_driver_race[["raceId", "driverId", "grid", "positionOrder", "driver_race_podium", "driver_race_outcome_known"]].copy()
q2_base = q2_base.merge(q2_races, on="raceId", how="left", validate="many_to_one")
q2_base = q2_base.merge(q2_circuits, on="circuitId", how="left", validate="many_to_one")
q2_base = q2_base.merge(q2_drivers, on="driverId", how="left", validate="many_to_one")
q2_base["year"] = pd.to_numeric(q2_base["year"], errors="coerce")
q2_base["grid"] = pd.to_numeric(q2_base["grid"], errors="coerce")
q2_base["positionOrder"] = pd.to_numeric(q2_base["positionOrder"], errors="coerce")
q2_base["podium"] = q2_base["driver_race_podium"].astype(bool)

# Explicit, readable mapping for nationality labels found in F1 records. Unlisted labels remain unmapped.
nationality_to_country = {
    "American": "United States", "Argentine": "Argentina", "Argentinian": "Argentina",
    "Australian": "Australia", "Austrian": "Austria", "Belgian": "Belgium", "Brazilian": "Brazil",
    "British": "United Kingdom", "Canadian": "Canada", "Chilean": "Chile", "Chinese": "China",
    "Colombian": "Colombia", "Czech": "Czech Republic", "Danish": "Denmark", "Dutch": "Netherlands",
    "East German": "Germany", "Finnish": "Finland", "French": "France", "German": "Germany",
    "Hungarian": "Hungary", "Indian": "India", "Indonesian": "Indonesia", "Irish": "Ireland",
    "Italian": "Italy", "Japanese": "Japan", "Liechtensteiner": "Liechtenstein", "Malaysian": "Malaysia",
    "Mexican": "Mexico", "Monegasque": "Monaco", "New Zealander": "New Zealand", "Polish": "Poland",
    "Portuguese": "Portugal", "Rhodesian": "Rhodesia", "Russian": "Russia", "South African": "South Africa",
    "Spanish": "Spain", "Swedish": "Sweden", "Swiss": "Switzerland", "Thai": "Thailand",
    "Uruguayan": "Uruguay", "Venezuelan": "Venezuela", "American-Italian": "United States"
}
country_aliases = {
    "uk": "United Kingdom", "u.k.": "United Kingdom", "united kingdom": "United Kingdom",
    "usa": "United States", "u.s.a.": "United States", "united states": "United States",
    "united states of america": "United States", "uae": "United Arab Emirates",
    "east germany": "Germany", "czech republic": "Czech Republic", "rhodesia": "Rhodesia"
}
def q2_normalize_country(value):
    if pd.isna(value): return pd.NA
    raw = str(value).strip()
    return country_aliases.get(raw.casefold(), raw)
def q2_map_nationality(value):
    if pd.isna(value): return pd.NA
    country = nationality_to_country.get(str(value).strip())
    return q2_normalize_country(country) if country is not None else pd.NA

q2_base["driver_country_normalized"] = q2_base["nationality"].map(q2_map_nationality)
q2_base["circuit_country_normalized"] = q2_base["circuit_country"].map(q2_normalize_country)
q2_base["home_away"] = pd.NA
q2_classifiable = q2_base["driver_country_normalized"].notna() & q2_base["circuit_country_normalized"].notna()
q2_base.loc[q2_classifiable, "home_away"] = np.where(
    q2_base.loc[q2_classifiable, "driver_country_normalized"].eq(q2_base.loc[q2_classifiable, "circuit_country_normalized"]), "Home", "Away")
q2_unknown = q2_base.loc[~q2_classifiable].copy()
q2_unknown["unmapped_reason"] = np.select(
    [q2_unknown["nationality"].isna(), q2_unknown["driver_country_normalized"].isna(), q2_unknown["circuit_country"].isna()],
    ["missing driver nationality", "nationality label not in explicit mapping", "missing circuit country"], default="country could not be normalized")
q2_known_outcome = q2_base.loc[q2_base["home_away"].notna() & q2_base["driver_race_outcome_known"]].copy()
q2_unadjusted = (q2_known_outcome.groupby("home_away", sort=True)
                 .agg(races=("podium", "size"), podiums=("podium", "sum"))
                 .reindex(["Home", "Away"]).reset_index())
q2_unadjusted["podium_rate"] = q2_unadjusted["podiums"] / q2_unadjusted["races"]

# Exact positive grid positions are the primary control; only grids observed in BOTH groups contribute.
q2_control = q2_known_outcome.loc[q2_known_outcome["grid"].gt(0)].copy()
q2_grid_excluded = len(q2_known_outcome) - len(q2_control)
q2_home_grid = (q2_control.loc[q2_control["home_away"].eq("Home")]
                .groupby("grid")["podium"].agg(home_starts="size", home_podiums="sum"))
q2_away_grid = (q2_control.loc[q2_control["home_away"].eq("Away")]
                .groupby("grid")["podium"].agg(away_starts="size", away_podiums="sum"))
q2_by_grid = q2_home_grid.join(q2_away_grid, how="inner").reset_index()
q2_by_grid["home_podium_rate"] = q2_by_grid["home_podiums"] / q2_by_grid["home_starts"]
q2_by_grid["away_podium_rate"] = q2_by_grid["away_podiums"] / q2_by_grid["away_starts"]
q2_by_grid["difference_home_minus_away"] = q2_by_grid["home_podium_rate"] - q2_by_grid["away_podium_rate"]
q2_by_grid["combined_starts"] = q2_by_grid["home_starts"] + q2_by_grid["away_starts"]
q2_by_grid["weight"] = q2_by_grid["combined_starts"] / q2_by_grid["combined_starts"].sum() if len(q2_by_grid) else np.nan
q2_adjusted_home = float((q2_by_grid["weight"] * q2_by_grid["home_podium_rate"]).sum()) if len(q2_by_grid) else np.nan
q2_adjusted_away = float((q2_by_grid["weight"] * q2_by_grid["away_podium_rate"]).sum()) if len(q2_by_grid) else np.nan
q2_adjusted_difference = q2_adjusted_home - q2_adjusted_away
q2_raw_home = float(q2_unadjusted.loc[q2_unadjusted["home_away"].eq("Home"), "podium_rate"].iloc[0])
q2_raw_away = float(q2_unadjusted.loc[q2_unadjusted["home_away"].eq("Away"), "podium_rate"].iloc[0])
q2_raw_difference = q2_raw_home - q2_raw_away
q2_summary = pd.DataFrame([{"classified_rows_home_away": len(q2_known_outcome), "unmatched_or_unknown_rows": len(q2_unknown),
    "valid_positive_grid_rows_classified": len(q2_control), "rows_excluded_from_control_for_nonpositive_or_missing_grid": q2_grid_excluded,
    "exact_grid_positions_with_both_groups": len(q2_by_grid), "unadjusted_home_rate": q2_raw_home, "unadjusted_away_rate": q2_raw_away,
    "unadjusted_difference_home_minus_away": q2_raw_difference, "grid_adjusted_home_rate": q2_adjusted_home,
    "grid_adjusted_away_rate": q2_adjusted_away, "adjusted_difference_home_minus_away": q2_adjusted_difference,
    "standardization_weight_sum": float(q2_by_grid["weight"].sum()) if len(q2_by_grid) else np.nan}])

q2_nationality_values = sorted(q2_base["nationality"].dropna().astype(str).unique().tolist())
q2_circuit_country_values = sorted(q2_base["circuit_country"].dropna().astype(str).unique().tolist())
print("Observed driver nationality labels:", q2_nationality_values)
print("Observed circuit country labels:", q2_circuit_country_values)
print(f"One row per driver-race; results_driver_race joined many-to-one to races, circuits and drivers. Rows with both countries classified home/away and valid outcome: {len(q2_known_outcome):,}.")
print(f"Rows unmatched/unknown and excluded from home-vs-away comparison: {len(q2_unknown):,}; missing nationality: {int(q2_base['nationality'].isna().sum()):,}; nationality labels not mapped: {int((q2_base['nationality'].notna() & q2_base['driver_country_normalized'].isna()).sum()):,}; missing circuit country: {int(q2_base['circuit_country'].isna().sum()):,}.")
q2_mapping_rows = []
for value in q2_nationality_values:
    q2_mapping_rows.append({"field": "driver nationality", "source_value": value, "normalized_country": q2_map_nationality(value), "rows": int(q2_base["nationality"].eq(value).sum())})
for value in q2_circuit_country_values:
    q2_mapping_rows.append({"field": "circuit country", "source_value": value, "normalized_country": q2_normalize_country(value), "rows": int(q2_base["circuit_country"].eq(value).sum())})
q2_country_mapping_audit = pd.DataFrame(q2_mapping_rows)
print("Country normalization audit:"); display(q2_country_mapping_audit)
print("Unadjusted descriptive comparison (context only; not the controlled answer):"); display(q2_unadjusted)
print(f"Positive-grid rows eligible for control: {len(q2_control):,}; excluded for missing/nonpositive grid: {q2_grid_excluded:,}; exact grid positions with both groups: {len(q2_by_grid):,}.")
print("Direct-standardization table by exact grid position:"); display(q2_by_grid)
print("Unadjusted and directly standardized summaries:"); display(q2_summary)

q2_country_mapping_audit.to_csv(STAGE2_EVIDENCE_DIR / "q2_country_mapping_audit.csv", index=False)
q2_unknown[["raceId", "driverId", "year", "nationality", "circuit_country", "unmapped_reason"]].to_csv(STAGE2_EVIDENCE_DIR / "q2_unknown_country_rows.csv", index=False)
q2_unadjusted.to_csv(STAGE2_EVIDENCE_DIR / "q2_home_away_unadjusted.csv", index=False)
q2_by_grid.to_csv(STAGE2_EVIDENCE_DIR / "q2_home_away_by_grid.csv", index=False)
q2_summary.to_csv(STAGE2_EVIDENCE_DIR / "q2_home_away_adjusted_summary.csv", index=False)

# Plot 1: visualization bands only; the controlled calculation above uses exact grid positions.
def q2_grid_band(grid):
    if grid <= 2: return "1-2"
    if grid <= 5: return "3-5"
    if grid <= 10: return "6-10"
    if grid <= 15: return "11-15"
    return "16+"
q2_control["grid_band"] = q2_control["grid"].map(q2_grid_band)
q2_band_order = ["1-2", "3-5", "6-10", "11-15", "16+"]
q2_band = (q2_control.groupby(["grid_band", "home_away"], observed=False)
           .agg(starts=("podium", "size"), podiums=("podium", "sum")).reset_index())
q2_band["podium_rate"] = q2_band["podiums"] / q2_band["starts"]
fig, ax = plt.subplots(figsize=(10, 6))
for label, color in [("Home", "darkorange"), ("Away", "steelblue")]:
    part = q2_band.loc[q2_band["home_away"].eq(label)].set_index("grid_band").reindex(q2_band_order)
    bars = ax.bar(np.arange(len(q2_band_order)) + (-0.18 if label == "Home" else 0.18), part["podium_rate"], width=0.36, label=label, color=color)
    for bar, (_, row) in zip(bars, part.iterrows()):
        if pd.notna(row["starts"]): ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.003, f"n={int(row['starts'])}", ha="center", fontsize=8)
ax.set_xticks(np.arange(len(q2_band_order)), q2_band_order); ax.yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0))
ax.set_title("Podium Rate by Starting-Grid Band (Visualization Only)"); ax.set_xlabel("Starting grid band"); ax.set_ylabel("Podium rate"); ax.legend(title="Nationality-country match")
fig.tight_layout(); fig.savefig(STAGE2_EVIDENCE_DIR / "q2_home_away_by_grid_band.png", dpi=160, bbox_inches="tight"); plt.show()

# Plot 2: unadjusted versus same-grid-distribution adjusted rates.
fig, ax = plt.subplots(figsize=(9, 6))
plot_labels = ["Unadjusted", "Grid-adjusted"]
for label, values, color in [("Home", [q2_raw_home, q2_adjusted_home], "darkorange"), ("Away", [q2_raw_away, q2_adjusted_away], "steelblue")]:
    bars = ax.bar(np.arange(2) + (-0.18 if label == "Home" else 0.18), values, width=0.36, label=label, color=color)
    for bar, val in zip(bars, values): ax.text(bar.get_x()+bar.get_width()/2, val+0.002, f"{val:.1%}", ha="center", fontsize=9)
ax.set_xticks(np.arange(2), plot_labels); ax.yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0))
ax.set_title("Home/Away Podium Rates Before and After Grid Standardization"); ax.set_xlabel("Comparison"); ax.set_ylabel("Podium rate"); ax.legend(title="Nationality-country match")
fig.tight_layout(); fig.savefig(STAGE2_EVIDENCE_DIR / "q2_raw_vs_grid_adjusted.png", dpi=160, bbox_inches="tight"); plt.show()

if abs(q2_adjusted_difference) < 0.001: q2_change_description = "the observed difference is near zero after adjustment"
elif q2_raw_difference == 0 or np.sign(q2_raw_difference) != np.sign(q2_adjusted_difference): q2_change_description = "the observed direction reverses after adjustment"
elif abs(q2_adjusted_difference) < abs(q2_raw_difference): q2_change_description = "the observed difference becomes smaller after adjustment"
else: q2_change_description = "the observed difference becomes larger after adjustment"
display(Markdown(f"Raw podium rates are {q2_raw_home:.2%} for home-country races and {q2_raw_away:.2%} for away races. After applying the same combined grid-position weights to both groups, the rates are {q2_adjusted_home:.2%} and {q2_adjusted_away:.2%}, respectively (adjusted difference {q2_adjusted_difference:+.2%}, home minus away); {q2_change_description}. This is an observational comparison, not a causal effect. Nationality is only the dataset's proxy for home country and does not encode residence, birthplace choice, dual nationality, or personal identity."))
print("Q2 evidence saved under:", STAGE2_EVIDENCE_DIR)
q2_missingness = pd.DataFrame({"field": ["year", "nationality", "circuit_country", "grid", "positionOrder"],
    "missing_rows": [int(q2_base[c].isna().sum()) for c in ["year", "nationality", "circuit_country", "grid", "positionOrder"]]})
q2_missingness.to_csv(STAGE2_EVIDENCE_DIR / "q2_missingness_audit.csv", index=False)
print("Q2 missingness audit before exclusions:"); display(q2_missingness)
print(f"Rows with a known country pair but missing positionOrder and excluded from rate denominators: {int((q2_classifiable & q2_base['positionOrder'].isna()).sum()):,}.")

## 2.3 Constructor mechanical-retirement rates

**Grain:** one cleaned constructor-driver-race entry. `results_driver_race` is joined to race year, constructor name, and status text with many-to-one validation. A start is any entry not explicitly labelled a non-start; a mechanical retirement requires an explicit car component or system failure. Generic “Retired” remains ambiguous and is not counted as mechanical.

In [ ]:
# 2.3 Constructor mechanical-retirement rates by regulation era
q3_races = clean_tables["races"][["raceId", "year"]].copy()
q3_constructors = clean_tables["constructors"][["constructorId", "name"]].rename(columns={"name": "constructor_name"})
q3_statuses = clean_tables["status"][["statusId", "status"]].rename(columns={"status": "status_text"})
q3_rows = results_driver_race[["raceId", "driverId", "constructorId", "statusId", "resultId"]].copy()
q3_rows = q3_rows.merge(q3_races, on="raceId", how="left", validate="many_to_one")
q3_rows = q3_rows.merge(q3_constructors, on="constructorId", how="left", validate="many_to_one")
q3_rows = q3_rows.merge(q3_statuses, on="statusId", how="left", validate="many_to_one")
q3_rows["year"] = pd.to_numeric(q3_rows["year"], errors="coerce")
q3_rows = q3_rows.loc[q3_rows["year"].between(2014, 2024)].copy()
q3_rows["era"] = np.where(q3_rows["year"].between(2014, 2021), "2014-2021", "2022-2024")

# Inspect and display every status actually present before applying classifications.
q3_status_frequency = (q3_rows.groupby(["statusId", "status_text"], dropna=False).size()
                       .rename("count_2014_2024").reset_index().sort_values(["count_2014_2024", "status_text"], ascending=[False, True], kind="mergesort"))
print(f"Q3 grain: one constructor-driver-race entry. Joined results_driver_race + races + constructors + status. Total 2014-2024 result rows: {len(q3_rows):,}.")
print("Complete 2014-2024 status frequency table (inspect before classification):")
display(q3_status_frequency)

# Exact whitelist: only explicit component/system failure labels qualify.
mechanical_status_whitelist_conservative = {
    "Engine", "Gearbox", "Transmission", "Clutch", "Hydraulics", "Electrical", "Electrics", "Electronics",
    "Suspension", "Brakes", "Differential", "Overheating", "Power Unit", "Turbo", "ERS", "Fuel pressure",
    "Fuel pump", "Fuel leak", "Oil pressure", "Oil leak", "Water pressure", "Water leak", "Cooling",
    "Cooling system", "Driveshaft", "Drive shaft", "Exhaust", "Wheel", "Puncture", "Tyre", "Tire",
    "Battery", "Alternator", "Ignition", "Radiator", "Steering", "Power loss", "Pneumatics", "ECU",
    "Fuel system", "Oil system", "Water system", "Throttle", "Chassis", "Spark plugs", "Water pump"
}
mechanical_status_whitelist = mechanical_status_whitelist_conservative | {"Mechanical", "Drivetrain", "Wheel nut"}
# Dataset status text identifies the failed component but not always its cause.
# Race 979 / resultId 23600 (2017 Hungary, Romain Grosjean) is documented as a
# wheel incorrectly fitted by the pit crew, so retain it as a start and exclude
# it from mechanical retirements. Race 963/23265 and 1056/25064 are included as
# wheel hardware/component failures.
q3_pit_crew_exception_result_ids = {23600}
nonstart_statuses = {"Did not start", "Did not qualify", "Did not prequalify", "Withdrew", "Withdrawn", "DNS", "Did not take part"}
regulation_statuses = {"Disqualified", "Excluded", "Black flagged", "Not eligible", "Regulation breach", "Technical infringement"}
incident_statuses = {"Accident", "Collision", "Spun off", "Injury", "Driver unwell", "Illness", "Damage", "Damage caused by collision", "Collision damage", "Accident damage"}
finished_statuses = {"Finished"}
import re
def q3_classify_status(status_text):
    if pd.isna(status_text): return ("OTHER / AMBIGUOUS", "missing status text")
    label = str(status_text).strip()
    low = label.casefold()
    if label in nonstart_statuses: return ("NON-START", "explicit non-start or withdrawal label")
    if label in regulation_statuses: return ("REGULATION / ADMINISTRATIVE", "explicit sporting/administrative decision")
    if label in incident_statuses or any(token in low for token in ["accident", "collision", "spun off", "driver unwell", "injury"]):
        return ("ACCIDENT / DRIVER INCIDENT", "incident or driver-related label")
    if label in mechanical_status_whitelist: return ("MECHANICAL", "exact component/system failure whitelist")
    if label in finished_statuses or re.fullmatch(r"\+\d+\s+Laps?", label, flags=re.IGNORECASE):
        return ("FINISHED / CLASSIFIED", "classified finish or lapped finisher")
    return ("OTHER / AMBIGUOUS", "not explicitly mechanical; retained as a start unless a non-start label")

q3_actual_status_labels = set(q3_status_frequency["status_text"].dropna().astype(str).unique())
q3_mechanical_statuses_used = sorted(q3_actual_status_labels & mechanical_status_whitelist)
q3_class_rows = []
for _, status_row in q3_status_frequency.iterrows():
    classification, reason = q3_classify_status(status_row["status_text"])
    q3_class_rows.append({"statusId": status_row["statusId"], "status_text": status_row["status_text"],
                          "count_2014_2024": int(status_row["count_2014_2024"]), "classification": classification, "classification_reason": reason})
q3_status_classification = pd.DataFrame(q3_class_rows)
q3_rows = q3_rows.merge(q3_status_classification[["statusId", "classification"]], on="statusId", how="left", validate="many_to_one")
q3_rows["classification_reason"] = q3_rows["statusId"].map(q3_status_classification.set_index("statusId")["classification_reason"])
q3_wheel_nut_exception = q3_rows["resultId"].isin(q3_pit_crew_exception_result_ids) & q3_rows["status_text"].eq("Wheel nut")
q3_rows.loc[q3_wheel_nut_exception, "classification"] = "PIT-CREW / OPERATIONAL"
q3_rows.loc[q3_wheel_nut_exception, "classification_reason"] = "Official race report identifies an incorrectly fitted wheel after pit stop; excluded from mechanical failures."
q3_rows["is_mechanical_retirement"] = q3_rows["classification"].eq("MECHANICAL")
q3_rows["is_mechanical_conservative"] = q3_rows["status_text"].isin(mechanical_status_whitelist_conservative)
q3_nonstarts = q3_rows["classification"].eq("NON-START")
q3_starts = q3_rows.loc[~q3_nonstarts].copy()
q3_ambiguous_all = int(q3_rows["classification"].eq("OTHER / AMBIGUOUS").sum())
q3_ambiguous_starts = int(q3_starts["classification"].eq("OTHER / AMBIGUOUS").sum())
q3_mechanical_count = int(q3_starts["is_mechanical_retirement"].sum())
q3_era_order = ["2014-2021", "2022-2024"]
q3_constructor_rates = (q3_starts.groupby(["era", "constructorId", "constructor_name"], dropna=False)
    .agg(starts=("driverId", "size"), mechanical_retirements=("is_mechanical_retirement", "sum")).reset_index())
q3_constructor_rates["mechanical_retirement_rate"] = q3_constructor_rates["mechanical_retirements"] / q3_constructor_rates["starts"]
q3_constructor_rates = q3_constructor_rates.sort_values(["era", "mechanical_retirement_rate", "starts", "constructor_name"],
    ascending=[True, False, False, True], kind="mergesort")
q3_constructor_rates["rank"] = q3_constructor_rates.groupby("era").cumcount() + 1
q3_era_comparison = q3_constructor_rates.pivot(index=["constructorId", "constructor_name"], columns="era",
    values=["starts", "mechanical_retirements", "mechanical_retirement_rate", "rank"])
q3_era_comparison.columns = [f"{metric}_{era}" for metric, era in q3_era_comparison.columns]
q3_era_comparison = q3_era_comparison.dropna(subset=["starts_2014-2021", "starts_2022-2024"]).reset_index()
q3_era_comparison["rate_change_2022_2024_minus_2014_2021_pp"] = 100 * (q3_era_comparison["mechanical_retirement_rate_2022-2024"] - q3_era_comparison["mechanical_retirement_rate_2014-2021"])
q3_status_classification.to_csv(STAGE2_EVIDENCE_DIR / "q3_status_classification.csv", index=False)
q3_status_frequency.to_csv(STAGE2_EVIDENCE_DIR / "q3_status_frequency.csv", index=False)
q3_record_classification = (q3_rows.groupby(["statusId", "status_text", "classification"], dropna=False)
    .size().rename("records").reset_index().sort_values(["status_text", "classification"], kind="mergesort"))
q3_record_classification.to_csv(STAGE2_EVIDENCE_DIR / "q3_record_level_classification.csv", index=False)
q3_conservative_starts = q3_starts.copy()
q3_conservative_counts = (q3_conservative_starts.groupby(["era", "constructorId", "constructor_name"], dropna=False)
    .agg(starts=("driverId", "size"), mechanical_retirements=("is_mechanical_conservative", "sum")).reset_index())
q3_conservative_counts["mechanical_retirement_rate"] = q3_conservative_counts["mechanical_retirements"] / q3_conservative_counts["starts"]
q3_corrected_compare = q3_constructor_rates.merge(q3_conservative_counts,
    on=["era", "constructorId", "constructor_name", "starts"], suffixes=("_corrected", "_conservative"), validate="one_to_one")
q3_corrected_compare.to_csv(STAGE2_EVIDENCE_DIR / "q3_conservative_vs_corrected_constructor_rates.csv", index=False)
q3_constructor_rates.to_csv(STAGE2_EVIDENCE_DIR / "q3_constructor_mechanical_rates.csv", index=False)
q3_era_comparison.to_csv(STAGE2_EVIDENCE_DIR / "q3_constructor_era_comparison.csv", index=False)
q3_mechanical_conservative_count = int(q3_starts["is_mechanical_conservative"].sum())
q3_mechanical_audit = pd.DataFrame([{"total_result_rows_2014_2024": len(q3_rows), "nonstart_rows_excluded": int(q3_nonstarts.sum()),
    "actual_starts": len(q3_starts), "mechanical_retirements": q3_mechanical_count,
    "conservative_mechanical_retirements": q3_mechanical_conservative_count,
    "verified_mechanical_additions": q3_mechanical_count - q3_mechanical_conservative_count,
    "pit_crew_exceptions_excluded": int(q3_wheel_nut_exception.sum()),
    "ambiguous_status_rows_all": q3_ambiguous_all, "ambiguous_status_rows_among_starts": q3_ambiguous_starts,
    "mechanical_statuses_used": "; ".join(sorted(q3_starts.loc[q3_starts["is_mechanical_retirement"], "status_text"].unique()))}])
q3_mechanical_audit.to_csv(STAGE2_EVIDENCE_DIR / "q3_start_and_mechanical_count_audit.csv", index=False)

print("Status classification audit (status-level labels):"); display(q3_status_classification)
print("Record-level status decisions, including the documented pit-crew exception:"); display(q3_record_classification)
print(f"Non-start rows excluded: {int(q3_nonstarts.sum()):,}; actual starts: {len(q3_starts):,}; conservative mechanical retirements: {q3_mechanical_conservative_count:,}; corrected mechanical retirements: {q3_mechanical_count:,}; pit-crew exception excluded: {int(q3_wheel_nut_exception.sum()):,}.")
print("Complete constructor-by-era rates:"); display(q3_constructor_rates)
q3_top_2014 = q3_constructor_rates.loc[q3_constructor_rates["era"].eq("2014-2021")].head(5)
q3_top_2022 = q3_constructor_rates.loc[q3_constructor_rates["era"].eq("2022-2024")].head(5)
print("Top five constructors per era; starts are denominators:"); display(pd.concat([q3_top_2014, q3_top_2022], ignore_index=True))
print("Constructor entities represented in both eras:"); display(q3_era_comparison)

# Horizontal bars show rates and event counts; complete ranked tables are in CSV.
fig, axes = plt.subplots(1, 2, figsize=(16, 8))
for ax, era_name in zip(axes, q3_era_order):
    part = q3_constructor_rates.loc[q3_constructor_rates["era"].eq(era_name)].head(10).sort_values("mechanical_retirement_rate")
    ax.barh(part["constructor_name"], part["mechanical_retirement_rate"], color="firebrick" if era_name == "2014-2021" else "slateblue")
    for y, (_, row) in enumerate(part.iterrows()): ax.text(row["mechanical_retirement_rate"] + 0.005, y, f"{int(row['mechanical_retirements'])}/{int(row['starts'])} starts", va="center", fontsize=8)
    ax.set_xlim(0, min(1.25, max(1.05, float(part["mechanical_retirement_rate"].max()) + 0.18)))
    ax.xaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0)); ax.set_title(era_name); ax.set_xlabel("Mechanical-retirement rate"); ax.set_ylabel("Constructor")
fig.suptitle("Constructor Mechanical-Retirement Rates by Era (Top 10; Complete Tables in CSV)"); fig.tight_layout(); fig.savefig(STAGE2_EVIDENCE_DIR / "q3_constructor_mechanical_rates_by_era.png", dpi=160, bbox_inches="tight"); plt.show()

def q3_top_text(frame):
    return "; ".join(f"{r.constructor_name} ({int(r.mechanical_retirements)}/{int(r.starts)}, {r.mechanical_retirement_rate:.1%})" for r in frame.head(3).itertuples())
q3_top1_text = q3_top_text(q3_top_2014)
q3_top2_text = q3_top_text(q3_top_2022)
print("Interpretation:")
display(Markdown("Highest ranked in 2014-2021: " + q3_top1_text + ". In 2022-2024: " + q3_top2_text + ". Rates use actual starts as denominators and constructorId entities are kept separate. Explicit Mechanical and Drivetrain labels are counted. Wheel-nut retirements are reviewed per record: component/hardware failures are counted, while the 2017 Hungarian incorrectly fitted wheel is excluded as a pit-crew operational error. Accidents, driver incidents, regulation decisions, non-starts and remaining ambiguous labels are excluded. The shorter recent era and differing constructor start counts limit direct comparisons; these are descriptive rates, not causal effects."))
print("Q3 evidence saved under:", STAGE2_EVIDENCE_DIR)

## 2.4 Stage 2 Validation

The checks below test the analysis grains, filters, aggregations, exact-grid overlap, standardization weights, status whitelist boundaries, and saved evidence outputs. Any failed assertion stops the run and identifies a condition to investigate.

In [ ]:
# 2.4 Compact programmatic validation of Q1–Q3 definitions, bounds, and evidence.
q1_rate_valid = q1_all_time['conversion_rate'].between(0, 1).all() and q1_by_era['conversion_rate'].between(0, 1).all()
q1_counts_valid = (q1_all_time['podiums_from_front_row'] <= q1_all_time['front_row_starts']).all() and (q1_by_era['podiums_from_front_row'] <= q1_by_era['front_row_starts']).all()
q1_totals_reconcile = int(q1_all_time['front_row_starts'].sum()) == len(q1_front) and int(q1_by_era['front_row_starts'].sum()) == len(q1_front)
q1_front_grids_valid = q1_front['grid'].isin([1, 2]).all()
q1_q2_use_corrected_driver_target = (
    np.array_equal(q1_base['podium'].astype('int8').to_numpy(), results_driver_race['driver_race_podium'].astype('int8').to_numpy())
    and np.array_equal(q2_base['podium'].astype('int8').to_numpy(), results_driver_race['driver_race_podium'].astype('int8').to_numpy()))
q1_era_labels_valid = set(q1_front['era'].dropna().unique()).issubset({'Pre-hybrid (1950-2013)', 'Hybrid (2014-2024)'})
q2_classification_valid = q2_base.loc[q2_classifiable, 'home_away'].isin(['Home', 'Away']).all()
q2_unknown_not_away = q2_base.loc[~q2_classifiable, 'home_away'].isna().all()
q2_exact_grid_overlap_valid = q2_by_grid['home_starts'].gt(0).all() and q2_by_grid['away_starts'].gt(0).all() and q2_by_grid['grid'].gt(0).all()
q2_rates_valid = q2_unadjusted['podium_rate'].between(0, 1).all() and q2_by_grid['home_podium_rate'].between(0, 1).all() and q2_by_grid['away_podium_rate'].between(0, 1).all()
q2_weights_valid = len(q2_by_grid) > 0 and np.isclose(q2_by_grid['weight'].sum(), 1.0)
q2_adjusted_rates_valid = np.isfinite([q2_adjusted_home, q2_adjusted_away]).all() and 0 <= q2_adjusted_home <= 1 and 0 <= q2_adjusted_away <= 1
q3_years_valid = q3_rows['year'].between(2014, 2024).all()
q3_eras_valid = set(q3_rows['era'].unique()).issubset({'2014-2021', '2022-2024'})
q3_nonstarts_excluded = not q3_starts['classification'].eq('NON-START').any()
q3_mechanical_counts_valid = q3_constructor_rates['mechanical_retirements'].le(q3_constructor_rates['starts']).all()
q3_rates_valid = q3_constructor_rates['mechanical_retirement_rate'].between(0, 1).all()
q3_statuses_classified = q3_status_classification['classification'].notna().all() and q3_status_classification['statusId'].nunique() == q3_status_frequency['statusId'].nunique()
q3_forbidden_statuses_not_mechanical = not q3_status_classification.loc[q3_status_classification['classification'].isin(['ACCIDENT / DRIVER INCIDENT', 'REGULATION / ADMINISTRATIVE', 'NON-START']), 'status_text'].isin(mechanical_status_whitelist).any()
q3_mechanical_whitelist_exact = q3_status_classification.loc[q3_status_classification['classification'].eq('MECHANICAL'), 'status_text'].isin(mechanical_status_whitelist).all()
stage1_pk_success = bool(pk_audit['passed'].all())
stage1_fk_success = bool((fk_audit['unmatched_rows'] == 0).all())
stage1_duplicate_years_success = duplicate_race_year_missing == 0 and duplicate_years_after == expected_duplicate_years
stage2_expected_evidence = [
    'q1_front_row_all_time.csv', 'q1_front_row_by_era.csv', 'q1_era_comparison.csv',
    'q1_qualifying_vs_grid_audit.csv', 'q1_all_time_top_circuits.png', 'q1_top_circuits_by_era.png',
    'q2_country_mapping_audit.csv', 'q2_home_away_unadjusted.csv', 'q2_home_away_by_grid.csv',
    'q2_home_away_adjusted_summary.csv', 'q2_home_away_by_grid_band.png', 'q2_raw_vs_grid_adjusted.png',
    'q3_status_frequency.csv', 'q3_status_classification.csv', 'q3_constructor_mechanical_rates.csv',
    'q3_constructor_era_comparison.csv', 'q3_start_and_mechanical_count_audit.csv',
    'q3_constructor_mechanical_rates_by_era.png'
]
stage2_evidence_exists = all((STAGE2_EVIDENCE_DIR / name).is_file() for name in stage2_expected_evidence)
stage2_checks = [
    ('Q1 rates within bounds', q1_rate_valid),
    ('Q1 podium counts do not exceed starts', q1_counts_valid),
    ('Q1 totals reconcile to eligible rows', q1_totals_reconcile),
    ('Q1 rows use front grid only', q1_front_grids_valid),
    ('Q1 and Q2 use corrected driver-race outcomes', q1_q2_use_corrected_driver_target),
    ('Q1 era labels valid', q1_era_labels_valid),
    ('Q2 classified nationality values valid', q2_classification_valid),
    ('Q2 unknown nationality remains unknown', q2_unknown_not_away),
    ('Q2 exact-grid overlap exists', q2_exact_grid_overlap_valid),
    ('Q2 rates within bounds', q2_rates_valid),
    ('Q2 standardization weights sum to one', q2_weights_valid),
    ('Q2 adjusted rates finite and bounded', q2_adjusted_rates_valid),
    ('Q3 years in scope', q3_years_valid),
    ('Q3 era labels valid', q3_eras_valid),
    ('Q3 non-starts excluded from starts', q3_nonstarts_excluded),
    ('Q3 mechanical retirements do not exceed starts', q3_mechanical_counts_valid),
    ('Q3 rates within bounds', q3_rates_valid),
    ('Every observed Q3 status has a classification', q3_statuses_classified),
    ('Q3 non-mechanical labels excluded from mechanical whitelist', q3_forbidden_statuses_not_mechanical),
    ('Q3 mechanical classification follows reviewed whitelist and record override', q3_mechanical_whitelist_exact and int(q3_wheel_nut_exception.sum()) == 1 and q3_mechanical_count == q3_mechanical_conservative_count + 10),
    ('Stage 1 PK constraints pass', stage1_pk_success),
    ('Stage 1 FK unmatched count is zero', stage1_fk_success),
    ('Stage 1 duplicate-year audit remains complete', stage1_duplicate_years_success),
    ('All required Stage 2 evidence exists', stage2_evidence_exists),
]
stage2_validation_summary = pd.DataFrame([{'check': name, 'passed': bool(ok)} for name, ok in stage2_checks])
assert len(stage2_validation_summary) == 24 and stage2_validation_summary['passed'].all(), 'Stage 2 validation failed.'
stage2_validation_summary.to_csv(STAGE2_EVIDENCE_DIR / 'stage2_validation_summary.csv', index=False)
display(stage2_validation_summary)
print(f"PASS checks: {int(stage2_validation_summary['passed'].sum())} / {len(stage2_validation_summary)}")
print('Stage 2 evidence files:', sorted(p.name for p in STAGE2_EVIDENCE_DIR.iterdir()))

# 3. Feature Selection, Engineering, Preprocessing & Dataset Splitting

Stage 3 builds an interpretable, leakage-controlled modeling table and documents feature selection, chronological splits, and train-only preprocessing. No predictive model is trained in Stage 3; predictive modeling begins in Stage 4.

## 3.1 Modeling Grain, Target & Leakage Rules
The target is a podium finish at the approved driver-race grain. Current-race results after the qualifying cutoff are excluded from predictors.

In [ ]:
# Stage 3 — full chronological feature engineering, EDA, split design, and preprocessing audit.
# Frozen Stage 1 and Stage 2 cells above remain unchanged; no classifier is fitted here.
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import sklearn

STAGE3_EVIDENCE_DIR = Path("/kaggle/working/stage3_feature_engineering_evidence")
STAGE3_EVIDENCE_DIR.mkdir(parents=True, exist_ok=True)

# 3.1 Prediction boundary and one-row-per-driver-race target.
stage3_keys = ["raceId", "driverId"]
assert isinstance(results_driver_race, pd.DataFrame)
assert set(stage3_keys + ["constructorId", "grid", "positionOrder", "driver_race_podium", "driver_race_outcome_known", "selected_entry_podium"]).issubset(results_driver_race.columns)
modeling_table = results_driver_race[["raceId", "driverId", "constructorId", "grid", "positionOrder", "driver_race_podium", "driver_race_outcome_known", "selected_entry_podium"]].copy()
modeling_table["positionOrder"] = pd.to_numeric(modeling_table["positionOrder"], errors="coerce")
modeling_table["podium"] = modeling_table["driver_race_podium"].astype("int8")
# 3.2 Base Modeling Table: validated race context joins.


## 3.2 Base Modeling Table
The table starts from the approved Stage 1 grain and adds race context with validated joins.


In [ ]:
# 3.2 Base Modeling Table: validated race context joins.
race_meta = clean_tables["races"][["raceId", "year", "round", "circuitId", "date"]].copy()
assert not race_meta["raceId"].duplicated().any(), "Race metadata must be unique by raceId."
modeling_table = modeling_table.merge(race_meta, on="raceId", how="left", validate="many_to_one")
modeling_table["year"] = pd.to_numeric(modeling_table["year"], errors="coerce")
modeling_table["round"] = pd.to_numeric(modeling_table["round"], errors="coerce")
modeling_table["circuitId"] = modeling_table["circuitId"].astype("Int64")
modeling_table["constructorId"] = pd.to_numeric(modeling_table["constructorId"], errors="coerce").astype("Int64")


## 3.3 Current-Race Grid Features
Positive starting positions are retained; zero, negative, and missing grids are represented as unknown.


In [ ]:
# 3.3 Current-Race Grid Features.
modeling_table["grid_numeric"] = pd.to_numeric(modeling_table["grid"], errors="coerce")
modeling_table["grid_position"] = modeling_table["grid_numeric"].where(modeling_table["grid_numeric"].gt(0))
modeling_table["grid_known"] = modeling_table["grid_position"].notna().astype("int8")
modeling_table = modeling_table.sort_values(["year", "round", "raceId", "driverId"], kind="mergesort").reset_index(drop=True)
assert len(modeling_table) == 26668, f"Expected the frozen Stage 1 grain of 26,668 rows; found {len(modeling_table):,}."
assert not modeling_table.duplicated(stage3_keys).any()



## 3.4 Current-Race Qualifying Features
Qualifying position and session gaps are joined without removing older races that lack qualifying coverage.


In [ ]:
# 3.4 Current-race qualifying features.
qualifying_source = clean_tables["qualifying"].copy()
qualifying_keys = ["raceId", "driverId"]
assert set(qualifying_keys + ["position", "q1_seconds", "q2_seconds", "q3_seconds"]).issubset(qualifying_source.columns)
qualifying_source = qualifying_source.dropna(subset=qualifying_keys).copy()
assert not qualifying_source.duplicated(qualifying_keys).any(), "Clean qualifying must be unique by (raceId, driverId)."
qual_features = qualifying_source[qualifying_keys + ["position", "q1_seconds", "q2_seconds", "q3_seconds"]].copy()
qual_features["qualifying_position"] = pd.to_numeric(qual_features["position"], errors="coerce")
for col in ["q1_seconds", "q2_seconds", "q3_seconds"]:
    qual_features[col] = pd.to_numeric(qual_features[col], errors="coerce").replace([np.inf, -np.inf], np.nan)
for session in ["q1", "q2", "q3"]:
    best = qual_features.groupby("raceId")[f"{session}_seconds"].transform("min")
    qual_features[f"{session}_gap_to_best"] = qual_features[f"{session}_seconds"] - best
qual_features["qualifying_available"] = qual_features["qualifying_position"].notna().astype("int8")
qual_features["q1_available"] = qual_features["q1_seconds"].notna().astype("int8")
qual_features["reached_q2"] = qual_features["q2_seconds"].notna().astype("int8")
qual_features["reached_q3"] = qual_features["q3_seconds"].notna().astype("int8")
qual_features = qual_features.drop(columns=["position"])
modeling_table = modeling_table.merge(qual_features, on=qualifying_keys, how="left", validate="one_to_one")
for col in ["qualifying_available", "q1_available", "reached_q2", "reached_q3"]:
    modeling_table[col] = modeling_table[col].fillna(0).astype("int8")
modeling_table["grid_minus_qualifying_position"] = modeling_table["grid_position"] - modeling_table["qualifying_position"]




## 3.5 Leakage-Safe Driver Historical Form
Driver history is shifted before expanding or rolling calculations, so each row uses only earlier races.


In [ ]:
# 3.5 Leakage-safe driver historical form.
modeling_table["driver_prior_starts"] = modeling_table.groupby("driverId", sort=False).cumcount()
modeling_table["driver_prior_podium_rate"] = modeling_table.groupby("driverId", sort=False)["podium"].transform(lambda s: s.shift(1).expanding(min_periods=1).mean())
modeling_table["driver_last5_podium_rate"] = modeling_table.groupby("driverId", sort=False)["podium"].transform(lambda s: s.shift(1).rolling(5, min_periods=1).mean())
modeling_table["driver_last5_avg_finish"] = modeling_table.groupby("driverId", sort=False)["positionOrder"].transform(lambda s: s.shift(1).rolling(5, min_periods=1).mean())
modeling_table["driver_has_history"] = modeling_table["driver_prior_starts"].gt(0).astype("int8")



## 3.6 Leakage-Safe Constructor Historical Form
Constructor outcomes are aggregated at constructor-race grain before shifted history is computed and merged back.


In [ ]:
# 3.6 Leakage-safe constructor historical form.
constructor_source = clean_tables["results"][["raceId", "constructorId", "driverId", "resultId", "positionOrder"]].copy()
constructor_source["positionOrder"] = pd.to_numeric(constructor_source["positionOrder"], errors="coerce")
constructor_source["constructor_entry_podium"] = constructor_source["positionOrder"].between(1, 3, inclusive="both").astype("int8")
constructor_driver_race = (constructor_source.groupby(["raceId", "constructorId", "driverId"], dropna=False, as_index=False)
    .agg(constructor_entry_podium=("constructor_entry_podium", "max"),
         constructor_entry_finish=("positionOrder", "mean")))
constructor_race_source = constructor_driver_race.merge(race_meta[["raceId", "year", "round"]], on="raceId", how="left", validate="many_to_one")
constructor_race = (constructor_race_source.groupby(["raceId", "constructorId", "year", "round"], dropna=False, as_index=False)
    .agg(constructor_entries=("driverId", "size"),
         constructor_podium_entries=("constructor_entry_podium", "sum"),
         constructor_race_avg_finish=("constructor_entry_finish", "mean")))
constructor_race = constructor_race.sort_values(["year", "round", "raceId", "constructorId"], kind="mergesort").reset_index(drop=True)
constructor_race["constructor_race_podium_rate"] = constructor_race["constructor_podium_entries"] / constructor_race["constructor_entries"]
constructor_group = constructor_race.groupby("constructorId", dropna=False, sort=False)
constructor_race["_prior_podium_entries"] = constructor_group["constructor_podium_entries"].transform(lambda s: s.shift(1).fillna(0).cumsum())
constructor_race["_prior_entries"] = constructor_group["constructor_entries"].transform(lambda s: s.shift(1).fillna(0).cumsum())
constructor_race["constructor_prior_podium_rate"] = constructor_race["_prior_podium_entries"].div(constructor_race["_prior_entries"].replace(0, np.nan))
constructor_race["_last5_podium_entries"] = constructor_group["constructor_podium_entries"].transform(lambda s: s.shift(1).rolling(5, min_periods=1).sum())
constructor_race["_last5_entries"] = constructor_group["constructor_entries"].transform(lambda s: s.shift(1).rolling(5, min_periods=1).sum())
constructor_race["constructor_last5_podium_rate"] = constructor_race["_last5_podium_entries"].div(constructor_race["_last5_entries"].replace(0, np.nan))
constructor_race["constructor_prior_races"] = constructor_race.groupby("constructorId", dropna=False, sort=False).cumcount()
constructor_race["constructor_last5_avg_finish"] = constructor_race.groupby("constructorId", dropna=False, sort=False)["constructor_race_avg_finish"].transform(lambda s: s.shift(1).rolling(5, min_periods=1).mean())
constructor_race["constructor_has_history"] = constructor_race["constructor_prior_races"].gt(0).astype("int8")
constructor_form_cols = ["raceId", "constructorId", "constructor_prior_races", "constructor_prior_podium_rate", "constructor_last5_podium_rate", "constructor_last5_avg_finish", "constructor_has_history"]
assert not constructor_race.duplicated(["raceId", "constructorId"]).any()
modeling_table = modeling_table.merge(constructor_race[constructor_form_cols], on=["raceId", "constructorId"], how="left", validate="many_to_one")



## 3.7 Previous-Race Championship Standings
Use each driver and constructor’s standings from the previous race in the same season, keeping current-race championship information out of predictors.

In [ ]:
# 3.7 Previous-race championship standings.
race_sequence = race_meta.sort_values(["year", "round", "raceId"], kind="mergesort").copy()
race_sequence["prev_raceId"] = race_sequence.groupby("year", sort=False)["raceId"].shift(1)
modeling_table = modeling_table.merge(race_sequence[["raceId", "prev_raceId"]], on="raceId", how="left", validate="many_to_one")
driver_stand = clean_tables["driver_standings"][["raceId", "driverId", "position", "points"]].copy()
driver_stand = driver_stand.rename(columns={"raceId": "driver_standing_source_raceId", "position": "driver_prev_standing_position", "points": "driver_prev_standing_points"})
driver_stand["driver_prev_standing_position"] = pd.to_numeric(driver_stand["driver_prev_standing_position"], errors="coerce")
driver_stand["driver_prev_standing_points"] = pd.to_numeric(driver_stand["driver_prev_standing_points"], errors="coerce")
driver_stand["driver_prev_leader_points"] = driver_stand.groupby("driver_standing_source_raceId")["driver_prev_standing_points"].transform("max")
driver_stand["driver_prev_points_vs_leader"] = driver_stand["driver_prev_standing_points"].div(driver_stand["driver_prev_leader_points"].where(driver_stand["driver_prev_leader_points"].gt(0)))
assert not driver_stand.duplicated(["driver_standing_source_raceId", "driverId"]).any(), "Prior driver standings must be unique by race and driver."
modeling_table = modeling_table.merge(driver_stand, left_on=["prev_raceId", "driverId"], right_on=["driver_standing_source_raceId", "driverId"], how="left", validate="many_to_one")
modeling_table["driver_prev_standing_available"] = modeling_table["driver_standing_source_raceId"].notna().astype("int8")
constructor_stand = clean_tables["constructor_standings"][["raceId", "constructorId", "position", "points"]].copy()
constructor_stand = constructor_stand.rename(columns={"raceId": "constructor_standing_source_raceId", "position": "constructor_prev_standing_position", "points": "constructor_prev_standing_points"})
constructor_stand["constructor_prev_standing_position"] = pd.to_numeric(constructor_stand["constructor_prev_standing_position"], errors="coerce")
constructor_stand["constructor_prev_standing_points"] = pd.to_numeric(constructor_stand["constructor_prev_standing_points"], errors="coerce")
constructor_stand["constructor_prev_leader_points"] = constructor_stand.groupby("constructor_standing_source_raceId")["constructor_prev_standing_points"].transform("max")
constructor_stand["constructor_prev_points_vs_leader"] = constructor_stand["constructor_prev_standing_points"].div(constructor_stand["constructor_prev_leader_points"].where(constructor_stand["constructor_prev_leader_points"].gt(0)))
assert not constructor_stand.duplicated(["constructor_standing_source_raceId", "constructorId"]).any(), "Prior constructor standings must be unique by race and constructor."
modeling_table = modeling_table.merge(constructor_stand, left_on=["prev_raceId", "constructorId"], right_on=["constructor_standing_source_raceId", "constructorId"], how="left", validate="many_to_one")
modeling_table["constructor_prev_standing_available"] = modeling_table["constructor_standing_source_raceId"].notna().astype("int8")



## 3.8 Feature Coverage & Selection EDA
Summarize predictor coverage and compare candidate raw feature groups on training-era data before freezing the final input set.

In [ ]:
# 3.8 Feature coverage and selection EDA.
train_mask = modeling_table["year"].le(2017)
eda_train = modeling_table.loc[train_mask].copy()
def stage3_savefig(fig, name):
    fig.tight_layout()
    fig.savefig(STAGE3_EVIDENCE_DIR / name, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close(fig)

fig, ax = plt.subplots(figsize=(11, 5))
grid_eda = eda_train.dropna(subset=["grid_position"]).groupby("grid_position").agg(starts=("podium", "size"), podium_rate=("podium", "mean"))
ax.plot(grid_eda.index, grid_eda["podium_rate"], marker="o", linewidth=1.4)
ax.set(xlabel="Positive starting grid position", ylabel="Podium rate", title="Training years (1950–2017): podium rate by starting grid")
stage3_savefig(fig, "stage3_train_podium_by_grid.png")

fig, ax = plt.subplots(figsize=(11, 5))
qual_eda = eda_train.dropna(subset=["qualifying_position"]).groupby("qualifying_position").agg(starts=("podium", "size"), podium_rate=("podium", "mean"))
ax.plot(qual_eda.index, qual_eda["podium_rate"], marker="o", linewidth=1.4)
ax.set(xlabel="Qualifying position", ylabel="Podium rate", title="Training years (1950–2017): podium rate by qualifying position")
stage3_savefig(fig, "stage3_train_podium_by_qualifying_position.png")

coverage_features = ["qualifying_position", "q1_gap_to_best", "q2_gap_to_best", "q3_gap_to_best", "driver_prev_standing_position", "constructor_prev_standing_position"]
yearly_feature_coverage = modeling_table.groupby("year")[coverage_features].agg(lambda s: 100 * s.notna().mean())
fig, ax = plt.subplots(figsize=(14, 7))
for feature in coverage_features:
    ax.plot(yearly_feature_coverage.index, yearly_feature_coverage[feature], marker=".", linewidth=1.4, label=feature)
ax.set(xlabel="Season", ylabel="Modeling rows with feature available (%)", title="Historical feature coverage by season (1950–2024)", ylim=(0, 105))
ax.grid(axis="y", alpha=.25)
ax.legend(ncol=2, fontsize=8, loc="upper left")
stage3_savefig(fig, "stage3_feature_coverage_by_year.png")
yearly_feature_coverage.rename_axis("year").to_csv(STAGE3_EVIDENCE_DIR / "stage3_feature_coverage_by_year.csv")

fig, ax = plt.subplots(figsize=(9, 5))
tmp = eda_train.dropna(subset=["driver_last5_podium_rate"]).copy()
if len(tmp) and tmp["driver_last5_podium_rate"].nunique() > 1:
    tmp["form_bin"] = pd.qcut(tmp["driver_last5_podium_rate"], q=min(5, tmp["driver_last5_podium_rate"].nunique()), duplicates="drop")
    dplot = tmp.groupby("form_bin", observed=False)["podium"].mean()
    ax.bar(range(len(dplot)), dplot.values, color="darkcyan")
    ax.set_xticks(range(len(dplot)), [str(x) for x in dplot.index], rotation=30, ha="right")
ax.set(xlabel="Prior five-start podium-rate quantile", ylabel="Current-race podium rate", title="Training years: driver past form vs podium outcome")
stage3_savefig(fig, "stage3_driver_form_bins_vs_podium.png")

fig, ax = plt.subplots(figsize=(9, 5))
tmp = eda_train.dropna(subset=["constructor_last5_podium_rate"]).copy()
if len(tmp) and tmp["constructor_last5_podium_rate"].nunique() > 1:
    tmp["form_bin"] = pd.qcut(tmp["constructor_last5_podium_rate"], q=min(5, tmp["constructor_last5_podium_rate"].nunique()), duplicates="drop")
    cplot = tmp.groupby("form_bin", observed=False)["podium"].mean()
    ax.bar(range(len(cplot)), cplot.values, color="darkorange")
    ax.set_xticks(range(len(cplot)), [str(x) for x in cplot.index], rotation=30, ha="right")
ax.set(xlabel="Prior five-race constructor podium-rate quantile", ylabel="Current-race podium rate", title="Training years: constructor past form vs podium outcome")
stage3_savefig(fig, "stage3_constructor_form_bins_vs_podium.png")

fig, ax = plt.subplots(figsize=(9, 5))
tmp = eda_train.dropna(subset=["driver_prev_standing_position"]).copy()
tmp["standing_band"] = pd.cut(tmp["driver_prev_standing_position"], bins=[0, 3, 10, np.inf], labels=["1–3", "4–10", "11+"])
splot = tmp.groupby("standing_band", observed=False)["podium"].mean()
ax.bar(splot.index.astype(str), splot.values, color="slateblue")
ax.set(xlabel="Position in previous same-season race standings", ylabel="Current-race podium rate", title="Training years: previous driver standing vs podium")
stage3_savefig(fig, "stage3_previous_driver_standing_vs_podium.png")

fig, ax = plt.subplots(figsize=(10, 8))
numeric_eda_cols = ["year", "round", "grid_position", "qualifying_position", "driver_prior_starts", "driver_prior_podium_rate", "driver_last5_podium_rate", "constructor_prior_races", "constructor_prior_podium_rate", "constructor_last5_podium_rate", "driver_prev_standing_position", "constructor_prev_standing_position", "podium"]
eda_corr = eda_train[numeric_eda_cols].corr(numeric_only=True)
sns.heatmap(eda_corr, cmap="coolwarm", center=0, ax=ax)
ax.set_title("Training years only: selected numeric-feature correlation")
stage3_savefig(fig, "stage3_train_numeric_correlation.png")

# Interpretations are deliberately descriptive and based only on years through 2017.
print("EDA interpretation (train years only): grid and qualifying plots show the historical association between stronger starting positions and podium frequency; EDA is used for feature review, not threshold tuning.")
print("Driver/constructor rolling-form plots compare podium frequency across bins formed from past-only rates. Previous-standing bands use immediately previous-race standings. Coverage varies substantially by era: qualifying-session timing is sparse in earlier seasons, and standings coverage also changes over time.")
print("Feature-selection decision: home_race is rejected because Stage 2 reported an exact-grid adjusted difference of -0.60 percentage points; no identity or post-race fields enter preprocessing.")



## 3.9 Final Selected Feature Set
Freeze the approved 31 raw predictors and exclude identifiers, post-race outcomes, and current-race information unavailable before the start.

In [ ]:
# 3.9 Final selected feature set; IDs and all current-race outcomes remain excluded.
feature_groups = {
    "context": ["year", "round", "circuitId", "constructorId"],
    "grid": ["grid_position", "grid_known"],
    "qualifying": ["qualifying_position", "qualifying_available", "q1_gap_to_best", "q2_gap_to_best", "q3_gap_to_best", "q1_available", "reached_q2", "reached_q3", "grid_minus_qualifying_position"],
    "driver_form": ["driver_prior_starts", "driver_prior_podium_rate", "driver_last5_podium_rate", "driver_last5_avg_finish", "driver_has_history"],
    "constructor_form": ["constructor_prior_races", "constructor_prior_podium_rate", "constructor_last5_podium_rate", "constructor_last5_avg_finish", "constructor_has_history"],
    "standings": ["driver_prev_standing_position", "driver_prev_points_vs_leader", "driver_prev_standing_available", "constructor_prev_standing_position", "constructor_prev_points_vs_leader", "constructor_prev_standing_available"]
}
selected_predictors = [feature for group in feature_groups.values() for feature in group]
assert len(selected_predictors) == len(set(selected_predictors)), "A selected feature must appear in exactly one feature group."
assert set(selected_predictors).issubset(modeling_table.columns)
feature_group_map = pd.DataFrame([{"feature": feature, "group": group, "selected": True} for group, features in feature_groups.items() for feature in features])
leakage_blacklist = {"resultId", "position", "positionText", "positionOrder", "points", "laps", "time", "milliseconds", "fastestLap", "rank", "fastestLapTime", "fastestLapSpeed", "statusId", "status", "pit_stop", "pit_stops", "lap_time", "lap_times", "driver_standings", "constructor_standings"}
leakage_columns = sorted(leakage_blacklist | {"raceId", "driverId", "prev_raceId", "driver_standing_source_raceId", "constructor_standing_source_raceId"})
feature_selection_decisions = pd.DataFrame([
    {"feature_group": group, "feature": feature, "pre_race_safe": True, "evidence": "EDA / past-only construction for " + group, "decision": "SELECTED", "reason": "Available by the post-qualifying, pre-race cutoff."}
    for group, features in feature_groups.items() for feature in features
] + [
    {"feature_group": "context", "feature": "home_race", "pre_race_safe": True, "evidence": "Stage 2 exact-grid adjusted home-away difference: -0.60 percentage points", "decision": "REJECTED", "reason": "Small, potentially confounded association; omit."},
    {"feature_group": "identity", "feature": "driverId", "pre_race_safe": True, "evidence": "High-cardinality identifier", "decision": "REJECTED", "reason": "Audit key only; not a generalizable predictor."},
    {"feature_group": "identity", "feature": "raceId", "pre_race_safe": True, "evidence": "Event identifier", "decision": "REJECTED", "reason": "Audit key only; year, round, and circuit context are selected."},
    {"feature_group": "current_race", "feature": "current positionOrder", "pre_race_safe": False, "evidence": "Observed after race start", "decision": "REJECTED", "reason": "Post-cutoff target outcome; leaks the podium label."},
    {"feature_group": "current_race", "feature": "current points", "pre_race_safe": False, "evidence": "Observed after race start", "decision": "REJECTED", "reason": "Post-cutoff result information."},
    {"feature_group": "current_race", "feature": "laps", "pre_race_safe": False, "evidence": "Race-progress measure", "decision": "REJECTED", "reason": "Unavailable before race start."},
    {"feature_group": "current_race", "feature": "status", "pre_race_safe": False, "evidence": "Race outcome classification", "decision": "REJECTED", "reason": "Unavailable before race start."},
    {"feature_group": "current_race", "feature": "race time", "pre_race_safe": False, "evidence": "Race result measure", "decision": "REJECTED", "reason": "Unavailable before race start."},
    {"feature_group": "current_race", "feature": "fastest-lap information", "pre_race_safe": False, "evidence": "Race-derived timing measure", "decision": "REJECTED", "reason": "Unavailable before race start."},
    {"feature_group": "current_standings", "feature": "current standings", "pre_race_safe": False, "evidence": "Includes current event", "decision": "REJECTED", "reason": "Use standings from the previous race only."},
    {"feature_group": "in_race", "feature": "pit stops and lap times", "pre_race_safe": False, "evidence": "Generated during a race", "decision": "REJECTED", "reason": "Unavailable at the prediction cutoff."},
    {"feature_group": "sprint", "feature": "current-race sprint results", "pre_race_safe": False, "evidence": "Not uniformly available at cutoff", "decision": "REJECTED", "reason": "Excluded from the consistent pre-race predictor set."}
])




## 3.10 Chronological Train / Validation / Test Split
Keep seasons ordered so validation and test rows represent later races than the training data.

In [ ]:
# 3.10 Chronological train / validation / test split.
modeling_table["split"] = np.select(
    [modeling_table["year"].le(2017), modeling_table["year"].between(2018, 2021), modeling_table["year"].between(2022, 2024)],
    ["train", "validation", "test"], default="out_of_scope")
assert modeling_table["split"].ne("out_of_scope").all()
split_summary = (modeling_table.groupby("split", sort=False).agg(rows=("podium", "size"), podiums=("podium", "sum"), podium_rate=("podium", "mean"), min_year=("year", "min"), max_year=("year", "max")).reindex(["train", "validation", "test"]).reset_index())
split_summary["non_podiums"] = split_summary["rows"] - split_summary["podiums"]
split_summary["years"] = split_summary["split"].map({"train": "1950–2017", "validation": "2018–2021", "test": "2022–2024"})
assert split_summary["min_year"].tolist() == [1950, 2018, 2022] and split_summary["max_year"].tolist() == [2017, 2021, 2024]
print("Chronological split summary (no rows dropped):")
display(split_summary)



## 3.11 Train-Fitted Preprocessing
Fit imputers, categorical encoding, and scaling on training rows only, then apply the frozen transformations to validation and test rows.

In [ ]:
# 3.11 Train-fitted preprocessing.
modeling_table["driver_standing_source_raceId"] = pd.to_numeric(modeling_table["driver_standing_source_raceId"], errors="coerce")
modeling_table["constructor_standing_source_raceId"] = pd.to_numeric(modeling_table["constructor_standing_source_raceId"], errors="coerce")
numeric_features = [f for f in selected_predictors if f not in ["circuitId", "constructorId"]]
categorical_features = ["circuitId", "constructorId"]
X_raw = modeling_table[selected_predictors].copy()
train_df = modeling_table.loc[modeling_table["split"].eq("train")].copy()
val_df = modeling_table.loc[modeling_table["split"].eq("validation")].copy()
test_df = modeling_table.loc[modeling_table["split"].eq("test")].copy()
for col in numeric_features:
    X_raw[col] = pd.to_numeric(X_raw[col], errors="coerce").astype("float64").replace([np.inf, -np.inf], np.nan)
for col in categorical_features:
    X_raw[col] = X_raw[col].astype("string").fillna("Missing").astype(str)
X_train_raw, X_val_raw, X_test_raw = X_raw.loc[train_df.index].copy(), X_raw.loc[val_df.index].copy(), X_raw.loc[test_df.index].copy()
y_train, y_val, y_test = train_df["podium"].copy(), val_df["podium"].copy(), test_df["podium"].copy()
train_rows = modeling_table["split"].eq("train").to_numpy()
validation_rows = modeling_table["split"].eq("validation").to_numpy()
test_rows = modeling_table["split"].eq("test").to_numpy()
try:
    numeric_imputer = SimpleImputer(strategy="median", keep_empty_features=True)
except TypeError:
    numeric_imputer = SimpleImputer(strategy="median")
try:
    encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)
preprocessor = ColumnTransformer(transformers=[
    ("numeric", Pipeline([("imputer", numeric_imputer), ("scaler", StandardScaler())]), numeric_features),
    ("categorical", Pipeline([("imputer", SimpleImputer(strategy="constant", fill_value="Missing")), ("onehot", encoder)]), categorical_features)
], remainder="drop", verbose_feature_names_out=False)
X_train_prepared = preprocessor.fit_transform(X_train_raw)
X_val_prepared = preprocessor.transform(X_val_raw)
X_test_prepared = preprocessor.transform(X_test_raw)
X_train_transformed = X_train_prepared
X_validation_transformed = X_val_prepared
X_test_transformed = X_test_prepared
assert X_train_prepared.shape == preprocessor.transform(X_train_raw).shape and np.allclose(X_train_prepared, preprocessor.transform(X_train_raw))
assert X_val_prepared.shape == preprocessor.transform(X_val_raw).shape and np.allclose(X_val_prepared, preprocessor.transform(X_val_raw))
assert X_test_prepared.shape == preprocessor.transform(X_test_raw).shape and np.allclose(X_test_prepared, preprocessor.transform(X_test_raw))
# Preprocessing-effect evidence uses actual TRAIN values and the fitted Stage 3 transformer.
numeric_imputer_fit = preprocessor.named_transformers_["numeric"].named_steps["imputer"]
numeric_train_matrix = np.asarray(X_train_transformed)[:, :len(numeric_features)]
preprocessing_effect_rows = []
for i, feature in enumerate(numeric_features):
    raw_train = pd.to_numeric(X_train_raw[feature], errors="coerce").replace([np.inf, -np.inf], np.nan)
    observed = raw_train.dropna()
    imputation_value = float(numeric_imputer_fit.statistics_[i])
    imputed_train = raw_train.fillna(imputation_value)
    scaled_train = pd.Series(numeric_train_matrix[:, i], dtype="float64")
    preprocessing_effect_rows.append({
        "feature": feature,
        "missing_before": int(raw_train.isna().sum()),
        "missing_after_imputation": int(imputed_train.isna().sum()),
        "imputation_value_train_median": imputation_value,
        "before_mean_raw_observed": float(observed.mean()) if len(observed) else np.nan,
        "before_std_raw_observed": float(observed.std(ddof=0)) if len(observed) else np.nan,
        "before_min_raw_observed": float(observed.min()) if len(observed) else np.nan,
        "before_max_raw_observed": float(observed.max()) if len(observed) else np.nan,
        "after_mean_scaled_train": float(scaled_train.mean()),
        "after_std_scaled_train": float(scaled_train.std(ddof=0)),
        "after_min_scaled_train": float(scaled_train.min()),
        "after_max_scaled_train": float(scaled_train.max()),
    })
preprocessing_effect_summary = pd.DataFrame(preprocessing_effect_rows)
assert preprocessing_effect_summary["missing_after_imputation"].eq(0).all()
assert np.isfinite(numeric_train_matrix).all()
preprocessing_effect_summary.to_csv(STAGE3_EVIDENCE_DIR / "stage3_preprocessing_effect_summary.csv", index=False)
# Show the numeric feature with the most missing TRAIN values and one with the fewest.
# The full numerical-feature evidence remains available in the CSV above.
_demo_order = preprocessing_effect_summary.sort_values(["missing_before", "feature"], ascending=[False, True], kind="mergesort")
preprocessing_demo_features = list(dict.fromkeys([_demo_order.iloc[0]["feature"], _demo_order.iloc[-1]["feature"]]))
display(Markdown("**Train-only preprocessing example.** Before statistics use observed raw TRAIN values (missing values skipped); after statistics use the fitted imputation-plus-scaling output for TRAIN. The median imputation value is learned from TRAIN only. Validation and test are transformed with this same fitted preprocessor; their values do not fit it."))
display(preprocessing_effect_summary.set_index("feature").loc[preprocessing_demo_features].round(4))
transformed_feature_names = list(preprocessor.get_feature_names_out())
assert X_train_transformed.shape[1] == len(transformed_feature_names)
assert X_validation_transformed.shape[1] == len(transformed_feature_names)
assert X_test_transformed.shape[1] == len(transformed_feature_names)
assert np.isfinite(X_train_transformed).all() and np.isfinite(X_validation_transformed).all() and np.isfinite(X_test_transformed).all()



## 3.12 Stage 3 Validation
Check modeling grain, predictor coverage, chronological splits, preprocessing boundaries, and saved Stage 3 evidence.

In [ ]:
# 3.12 Evidence outputs and validation.
modeling_table["split"] = modeling_table["split"].astype(str)
model_table_columns = ["raceId", "driverId", "year", "round", "date", "podium"] + selected_predictors + ["prev_raceId", "driver_standing_source_raceId", "constructor_standing_source_raceId", "split"]
modeling_table_stage3 = modeling_table[model_table_columns].copy()
modeling_table_stage3["split"] = modeling_table_stage3["split"].map({"train": "Train", "validation": "Validation", "test": "Test"})
modeling_table_stage3.to_csv(STAGE3_EVIDENCE_DIR / "modeling_table_stage3.csv", index=False)
modeling_table_summary = pd.DataFrame([{"metric": "modeling rows", "value": len(modeling_table)}, {"metric": "unique (raceId, driverId) rows", "value": modeling_table[stage3_keys].drop_duplicates().shape[0]}, {"metric": "target podium rows", "value": int(modeling_table["podium"].sum())}, {"metric": "target non-podium rows", "value": int((1-modeling_table["podium"]).sum())}, {"metric": "podium prevalence", "value": float(modeling_table["podium"].mean())}, {"metric": "year range", "value": str(int(modeling_table["year"].min())) + "–" + str(int(modeling_table["year"].max()))}, {"metric": "selected predictors", "value": len(selected_predictors)}, {"metric": "numerical predictors", "value": len(numeric_features)}, {"metric": "categorical predictors", "value": len(categorical_features)}, {"metric": "transformed features", "value": len(transformed_feature_names)}, {"metric": "years", "value": "1950–2024"}, {"metric": "prediction cutoff", "value": "after qualifying, before race start"}, {"metric": "models trained", "value": 0}])
modeling_table_summary.to_csv(STAGE3_EVIDENCE_DIR / "stage3_modeling_table_summary.csv", index=False)
feature_dictionary = pd.DataFrame([{"feature": f, "group": feature_group_map.set_index("feature").loc[f, "group"], "dtype": str(modeling_table[f].dtype), "missing_rows": int(modeling_table[f].isna().sum()), "role": "categorical" if f in categorical_features else "numeric", "definition": ("Missing when unknown; categorical identity used as pre-race context." if f in categorical_features else "Selected pre-race predictor; historical features use shifted prior events only.")} for f in selected_predictors])
feature_dictionary.to_csv(STAGE3_EVIDENCE_DIR / "stage3_feature_dictionary.csv", index=False)
feature_selection_decisions.to_csv(STAGE3_EVIDENCE_DIR / "stage3_feature_selection_decisions.csv", index=False)
feature_group_map.to_csv(STAGE3_EVIDENCE_DIR / "stage3_feature_group_map.csv", index=False)
split_summary.to_csv(STAGE3_EVIDENCE_DIR / "stage3_split_summary.csv", index=False)
missingness_rows = []
for split_name in ["train", "validation", "test"]:
    part = modeling_table.loc[modeling_table["split"].eq(split_name), selected_predictors]
    for feature in selected_predictors:
        miss = int(part[feature].isna().sum())
        missingness_rows.append({"split": split_name, "feature": feature, "rows": len(part), "missing_rows": miss, "missing_rate": miss / len(part) if len(part) else np.nan})
missingness_by_split = pd.DataFrame(missingness_rows)
missingness_by_split.to_csv(STAGE3_EVIDENCE_DIR / "stage3_missingness_by_split.csv", index=False)
preprocessing_summary = pd.DataFrame([
    {"split": "train", "rows": int(train_rows.sum()), "transformed_features": X_train_transformed.shape[1], "fit_used": True},
    {"split": "validation", "rows": int(validation_rows.sum()), "transformed_features": X_validation_transformed.shape[1], "fit_used": False},
    {"split": "test", "rows": int(test_rows.sum()), "transformed_features": X_test_transformed.shape[1], "fit_used": False}
])
preprocessing_summary["numerical_features"] = len(numeric_features)
preprocessing_summary["categorical_features"] = len(categorical_features)
preprocessing_summary["sklearn_version"] = sklearn.__version__
preprocessing_summary.to_csv(STAGE3_EVIDENCE_DIR / "stage3_preprocessing_summary.csv", index=False)
pd.DataFrame({"transformed_feature": transformed_feature_names}).to_csv(STAGE3_EVIDENCE_DIR / "stage3_transformed_feature_names.csv", index=False)


# Stage 3 frozen-input validation: feature-level checks plus split and transform invariants.
stage3_checks = []
def add_s3(name, ok): stage3_checks.append({'check': name, 'passed': bool(ok)})
add_s3('Modeling table retains 26,668 rows', len(modeling_table) == 26668)
add_s3('Modeling grain is unique by race and driver', not modeling_table.duplicated(stage3_keys).any())
add_s3('Six approved feature groups are present', set(feature_groups) == {'context','grid','qualifying','driver_form','constructor_form','standings'})
add_s3('Chronological split sizes match approved counts', [len(train_df),len(val_df),len(test_df)] == [23689,1620,1359])
add_s3('Chronological split year ranges match approved windows', (train_df.year.min(),train_df.year.max(),val_df.year.min(),val_df.year.max(),test_df.year.min(),test_df.year.max()) == (1950,2017,2018,2021,2022,2024))
add_s3('All transformed matrices have 306 named features', X_train_transformed.shape[1] == X_validation_transformed.shape[1] == X_test_transformed.shape[1] == len(transformed_feature_names) == 306)
for feature in selected_predictors:
    add_s3(f'Selected feature {feature} exists in the modeling table', feature in modeling_table.columns)
    add_s3(f'Selected feature {feature} appears once in the feature groups', sum(feature in members for members in feature_groups.values()) == 1)
for group, members in feature_groups.items():
    add_s3(f'Feature group {group} is nonempty and mapped correctly', len(members)>0 and set(feature_group_map.loc[feature_group_map.group.eq(group),'feature']) == set(members))

driver_form_reference = modeling_table.groupby("driverId", sort=False)["podium"].transform(lambda s: s.shift(1).expanding(min_periods=1).mean())
driver_form_is_corrected_and_past_only = np.allclose(modeling_table["driver_prior_podium_rate"].fillna(-1), driver_form_reference.fillna(-1))
constructor_outcome_rows_match_source = (
    int(constructor_race["constructor_podium_entries"].sum()) == int(constructor_driver_race["constructor_entry_podium"].sum())
    and not constructor_race.duplicated(["raceId", "constructorId"]).any())

# Synthetic cross-constructor shared-drive regression case: the driver earns a
# podium in constructor 20, while the pre-race grid-selected entry is constructor
# 10 and finishes outside the podium. The driver label is positive; constructor
# 10's entry outcome remains negative; constructor 20 retains its own podium.
synthetic_shared = pd.DataFrame([
    {"raceId": 900001, "driverId": 700001, "constructorId": 10, "grid": 1, "positionOrder": 12},
    {"raceId": 900001, "driverId": 700001, "constructorId": 20, "grid": 9, "positionOrder": 2},
])
synthetic_shared["entry_podium"] = synthetic_shared["positionOrder"].between(1, 3)
synthetic_driver_target = bool(synthetic_shared["entry_podium"].any())
synthetic_selected = synthetic_shared.sort_values(["grid"], kind="mergesort").iloc[0]
synthetic_ctor_outcomes = synthetic_shared.groupby("constructorId")["entry_podium"].any().to_dict()
constructor_attribution_test_passed = (
    synthetic_driver_target and int(synthetic_selected["constructorId"]) == 10
    and not bool(synthetic_selected["entry_podium"])
    and synthetic_ctor_outcomes == {10: False, 20: True})
add_s3('Driver historical podium form uses corrected prior driver-race outcomes only', driver_form_is_corrected_and_past_only)
add_s3('Constructor history uses constructor-attributed source results', constructor_outcome_rows_match_source)
add_s3('Synthetic shared-drive outcome is not credited to selected constructor', constructor_attribution_test_passed)

stage3_validation_summary = pd.DataFrame(stage3_checks)
assert len(stage3_validation_summary)==77, f'Expected 77 Stage 3 checks; found {len(stage3_validation_summary)}.'
assert stage3_validation_summary.passed.all(), 'Stage 3 validation failed.'
stage3_validation_summary.to_csv(STAGE3_EVIDENCE_DIR/'stage3_validation_summary.csv', index=False)
display(stage3_validation_summary)
print(f"PASS checks: {int(stage3_validation_summary.passed.sum())} / {len(stage3_validation_summary)}")


# 4. Predictive Modeling
Stage 4 compares the frozen model families on identical Stage 3 predictors and chronological rows. Hyperparameters and thresholds are selected on training/validation only; test evaluation follows an explicit freeze.

All candidates use the same 31 raw predictors, the Stage 3 train-fitted preprocessing schema (306 transformed columns), and the chronological train (1950?2017), validation (2018?2021), and test (2022?2024) partitions. Logistic Regression models a linear log-odds relationship; the Random Forest combines decision trees to capture nonlinear splits and interactions. NN_3 is the selected shallow neural network: 306 inputs, Dense hidden layers of 64 and 32 ReLU units with dropout rates 0.20 and 0.10, then a sigmoid podium-probability output. Its balanced class weights are computed from training labels only, and early stopping monitors validation PR-AUC. Candidate/model ranking uses validation PR-AUC (average precision), with ROC-AUC as a tie-break; each selected model?s decision threshold is chosen separately by validation F1.

## 4.1 Evaluation Protocol & Metrics
Set deterministic seeds, define shared metrics, and use the already fitted Stage 3 transformer without refitting. This establishes one consistent evaluation contract for all candidate models.

In [ ]:
# 4.1 Shared evaluation helpers: Stage 3 preprocessor remains frozen.
import os, random, json, warnings, joblib, sklearn
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score, precision_score, recall_score, accuracy_score, confusion_matrix, roc_curve, precision_recall_curve)
from sklearn.utils.class_weight import compute_class_weight
from IPython.display import display, Markdown
import tensorflow as tf
RANDOM_STATE=42
np.random.seed(RANDOM_STATE); random.seed(RANDOM_STATE); tf.keras.utils.set_random_seed(RANDOM_STATE)
try: tf.config.experimental.enable_op_determinism()
except Exception: pass
STAGE4_EVIDENCE_DIR=Path('/kaggle/working/stage4_modeling_evidence'); STAGE4_EVIDENCE_DIR.mkdir(parents=True,exist_ok=True)
X_train_s4, X_val_s4 = X_train_prepared, X_val_prepared
X_train_nn=np.asarray(X_train_prepared,dtype=np.float32); X_val_nn=np.asarray(X_val_prepared,dtype=np.float32)
y_train_s4=np.asarray(y_train,dtype=int).ravel(); y_val_s4=np.asarray(y_val,dtype=int).ravel()
feature_count_s4=int(X_train_prepared.shape[1])
assert len(y_train_s4)==X_train_prepared.shape[0] and len(y_val_s4)==X_val_prepared.shape[0]
assert X_train_prepared.shape[1]==X_val_prepared.shape[1]==306

def stage4_metrics(y_true, probability, threshold=0.5):
    y_true=np.asarray(y_true,dtype=int).ravel(); probability=np.asarray(probability,dtype=float).ravel()
    assert len(y_true)==len(probability) and np.isfinite(probability).all() and ((probability>=0)&(probability<=1)).all()
    pred=(probability>=float(threshold)).astype(int)
    result={'roc_auc':float(roc_auc_score(y_true,probability)),'pr_auc':float(average_precision_score(y_true,probability)),
      'f1':float(f1_score(y_true,pred,zero_division=0)),'precision':float(precision_score(y_true,pred,zero_division=0)),
      'recall':float(recall_score(y_true,pred,zero_division=0)),'accuracy':float(accuracy_score(y_true,pred)),
      'threshold':float(threshold),'positive_predictions':int(pred.sum())}
    assert np.isfinite(list(result.values())).all()
    return result


## 4.2 Logistic Regression Baseline & Selection
Fit six prespecified candidates on training rows only. Rank by validation PR-AUC, then ROC-AUC, then smaller C; retain the unweighted C=1 baseline for comparison.


In [ ]:
# 4.2 Logistic candidate fits and validation-only selection.
logistic_rows=[]; logistic_models={}; logistic_probabilities={}
for c_value in [0.1,1.0,10.0]:
    for cw in [None,'balanced']:
        name=f"LR_C{c_value:g}_"+('balanced' if cw else 'unweighted')
        model=LogisticRegression(C=c_value,class_weight=cw,solver='liblinear',max_iter=2000,random_state=RANDOM_STATE)
        model.fit(X_train_s4,y_train_s4)
        pt=model.predict_proba(X_train_s4)[:,1]; pv=model.predict_proba(X_val_s4)[:,1]
        mt=stage4_metrics(y_train_s4,pt); mv=stage4_metrics(y_val_s4,pv)
        logistic_models[name]=model; logistic_probabilities[name]={'train':pt,'validation':pv}
        logistic_rows.append({'candidate':name,'C':c_value,'class_weight':cw or 'None','train_roc_auc':mt['roc_auc'],'train_pr_auc':mt['pr_auc'],'validation_roc_auc':mv['roc_auc'],'validation_pr_auc':mv['pr_auc']})
stage4_logistic_candidates=pd.DataFrame(logistic_rows)
stage4_logistic_candidates.to_csv(STAGE4_EVIDENCE_DIR/'stage4_logistic_candidates.csv',index=False)
logistic_baseline_name='LR_C1_unweighted'
logistic_baseline_row=stage4_logistic_candidates.loc[stage4_logistic_candidates.candidate.eq(logistic_baseline_name)].iloc[0]
logistic_selected_row=stage4_logistic_candidates.sort_values(['validation_pr_auc','validation_roc_auc','C'],ascending=[False,False,True],kind='mergesort').iloc[0]
selected_logistic_name=str(logistic_selected_row.candidate); selected_logistic_model=logistic_models[selected_logistic_name]
display(stage4_logistic_candidates)


## 4.3 Random Forest Selection
Fit four fixed configurations on training rows only. Select with validation PR-AUC, then ROC-AUC, then candidate order.


In [ ]:
# 4.3 Fixed Random Forest candidates, ranked on validation only.
rf_specs=[('RF_1',None,1,None),('RF_2',16,2,'balanced_subsample'),('RF_3',24,2,'balanced_subsample'),('RF_4',16,5,'balanced_subsample')]
rf_rows=[]; rf_models={}; rf_probabilities={}
for order,(name,depth,leaf,cw) in enumerate(rf_specs):
    model=RandomForestClassifier(n_estimators=400,max_depth=depth,min_samples_leaf=leaf,class_weight=cw,random_state=RANDOM_STATE,n_jobs=-1)
    model.fit(X_train_prepared,y_train_s4)
    pt=model.predict_proba(X_train_prepared)[:,1]; pv=model.predict_proba(X_val_prepared)[:,1]
    mt=stage4_metrics(y_train_s4,pt); mv=stage4_metrics(y_val_s4,pv)
    rf_models[name]=model; rf_probabilities[name]={'train':pt,'validation':pv}
    rf_rows.append({'candidate':name,'order':order+1,'n_estimators':400,'max_depth':depth,'min_samples_leaf':leaf,'class_weight':cw or 'None','train_roc_auc':mt['roc_auc'],'train_pr_auc':mt['pr_auc'],'validation_roc_auc':mv['roc_auc'],'validation_pr_auc':mv['pr_auc']})
stage4_random_forest_candidates=pd.DataFrame(rf_rows)
stage4_random_forest_candidates.to_csv(STAGE4_EVIDENCE_DIR/'stage4_random_forest_candidates.csv',index=False)
rf_selected_row=stage4_random_forest_candidates.sort_values(['validation_pr_auc','validation_roc_auc','order'],ascending=[False,False,True],kind='mergesort').iloc[0]
selected_rf_name=str(rf_selected_row.candidate); selected_rf_model=rf_models[selected_rf_name]
display(stage4_random_forest_candidates)


## 4.4 Shallow Neural Network Selection
Compare one- and two-layer networks on identical train/validation rows. Early stopping monitors validation PR-AUC; balanced class weights use training labels only.


In [ ]:
# 4.4 Three shallow Keras candidates; class weighting uses training labels only.
class_values=np.unique(y_train_s4); cw_values=compute_class_weight(class_weight='balanced',classes=class_values,y=y_train_s4)
train_class_weights={int(c):float(w) for c,w in zip(class_values,cw_values)}
X_train_nn=np.asarray(X_train_prepared,dtype=np.float32); X_val_nn=np.asarray(X_val_prepared,dtype=np.float32)
nn_specs=[{'candidate':'NN_1','layers':[64],'dropout':[.20],'class_weight':'none'}, {'candidate':'NN_2','layers':[64],'dropout':[.20],'class_weight':'balanced'}, {'candidate':'NN_3','layers':[64,32],'dropout':[.20,.10],'class_weight':'balanced'}]
nn_models={}; nn_probs={}; nn_histories={}; nn_rows=[]
for spec in nn_specs:
    tf.keras.backend.clear_session(); tf.keras.utils.set_random_seed(RANDOM_STATE)
    inp=tf.keras.Input(shape=(X_train_nn.shape[1],)); z=inp
    for i,(units,drop) in enumerate(zip(spec['layers'],spec['dropout'])):
        z=tf.keras.layers.Dense(units,activation='relu',name=f'hidden_{i+1}')(z); z=tf.keras.layers.Dropout(drop,name=f'dropout_{i+1}')(z)
    out=tf.keras.layers.Dense(1,activation='sigmoid',name='podium_probability')(z); model=tf.keras.Model(inp,out)
    model.compile(optimizer=tf.keras.optimizers.Adam(),loss='binary_crossentropy',metrics=[tf.keras.metrics.AUC(curve='ROC',name='roc_auc'),tf.keras.metrics.AUC(curve='PR',name='pr_auc')])
    callback=tf.keras.callbacks.EarlyStopping(monitor='val_pr_auc',mode='max',patience=8,restore_best_weights=True)
    fitkw={'class_weight':train_class_weights} if spec['class_weight']=='balanced' else {}
    hist=model.fit(X_train_nn,y_train_s4,validation_data=(X_val_nn,y_val_s4),epochs=100,batch_size=128,verbose=0,callbacks=[callback],**fitkw)
    history={k:[float(v) for v in vals] for k,vals in hist.history.items()}; ptrain=model.predict(X_train_nn,verbose=0).ravel(); pval=model.predict(X_val_nn,verbose=0).ravel()
    mt=stage4_metrics(y_train_s4,ptrain); mv=stage4_metrics(y_val_s4,pval)
    nnn=spec['candidate']; nn_models[nnn]=model; nn_probs[nnn]={'train':ptrain,'validation':pval}; nn_histories[nnn]=history
    nn_rows.append({'candidate':nnn,'hidden_layers':json.dumps(spec['layers']),'dropout':json.dumps(spec['dropout']),'class_weight':spec['class_weight'],'class_weights_train':json.dumps(train_class_weights),'epochs_trained':len(history['loss']),'best_epoch':int(np.argmax(history['val_pr_auc'])+1),'best_validation_pr_auc_history':max(history['val_pr_auc']),'best_weights_restored':bool(callback.restore_best_weights),'train_roc_auc':mt['roc_auc'],'train_pr_auc':mt['pr_auc'],'validation_roc_auc':mv['roc_auc'],'validation_pr_auc':mv['pr_auc']})
stage4_nn_candidates=pd.DataFrame(nn_rows); stage4_nn_candidates.to_csv(STAGE4_EVIDENCE_DIR/'stage4_nn_candidates.csv',index=False)
nn_selected_row=stage4_nn_candidates.sort_values(['validation_pr_auc','validation_roc_auc'],ascending=False,kind='mergesort').iloc[0]
selected_nn_name=str(nn_selected_row.candidate); selected_nn_model=nn_models[selected_nn_name]
display(stage4_nn_candidates)


## 4.5 Validation-Based Threshold Selection
Choose one deterministic threshold per selected model by maximizing validation F1. Ties prefer the value closest to 0.50, then the lower value.


In [ ]:
# 4.5 One validation-only F1 threshold per selected family.
selected_candidate_names={'Logistic Regression':selected_logistic_name,'Random Forest':selected_rf_name,'Shallow Neural Network':selected_nn_name}
selected_models={'Logistic Regression':selected_logistic_model,'Random Forest':selected_rf_model,'Shallow Neural Network':selected_nn_model}
train_probs={'Logistic Regression':logistic_probabilities[selected_logistic_name]['train'],'Random Forest':rf_probabilities[str(rf_selected_row.candidate)]['train'],'Shallow Neural Network':nn_probs[selected_nn_name]['train']}
val_probs={'Logistic Regression':logistic_probabilities[selected_logistic_name]['validation'],'Random Forest':rf_probabilities[str(rf_selected_row.candidate)]['validation'],'Shallow Neural Network':nn_probs[selected_nn_name]['validation']}
selected_thresholds={}; threshold_rows=[]
for name in selected_models:
    p=val_probs[name]; cand=np.linspace(.01,.99,99); scored=[(float(f1_score(y_val_s4,p>=t,zero_division=0)),float(t)) for t in cand]
    best=max(v for v,t in scored); tied=[t for v,t in scored if np.isclose(v,best,atol=1e-12,rtol=0)]; thr=sorted(tied,key=lambda t:(abs(t-.5),t))[0]
    selected_thresholds[name]=thr; default=stage4_metrics(y_val_s4,p,.5); chosen=stage4_metrics(y_val_s4,p,thr)
    threshold_rows.append({'model':name,'candidate':selected_candidate_names[name],'default_f1':default['f1'],'threshold':thr,'validation_f1':chosen['f1'],'precision':chosen['precision'],'recall':chosen['recall'],'selection_set':'validation'})
stage4_threshold_selection=pd.DataFrame(threshold_rows); stage4_threshold_selection.to_csv(STAGE4_EVIDENCE_DIR/'stage4_threshold_selection.csv',index=False)
display(stage4_threshold_selection)


## 4.6 Final Training / Validation Comparison
Report all requested threshold-based and ranking metrics on training and validation. Train-minus-validation gaps are descriptive and not a reason to tune on test.


In [ ]:
# 4.6 Final train/validation metrics and overfitting gaps.
trainval_rows=[]
for model_name in selected_models:
    for split_name,truth,prob in [('train',y_train_s4,train_probs[model_name]),('validation',y_val_s4,val_probs[model_name])]:
        trainval_rows.append({'model':model_name,'candidate':selected_candidate_names[model_name],'split':split_name,**stage4_metrics(truth,prob,selected_thresholds[model_name])})
stage4_train_validation_metrics=pd.DataFrame(trainval_rows)
stage4_overfitting_audit=[]
for model_name in selected_models:
    tr=stage4_train_validation_metrics.query('model == @model_name and split == "train"').iloc[0]; va=stage4_train_validation_metrics.query('model == @model_name and split == "validation"').iloc[0]
    stage4_overfitting_audit.append({'model':model_name,**{f'train_minus_validation_{m}':float(tr[m]-va[m]) for m in ['roc_auc','pr_auc','f1','precision','recall','accuracy']}})
stage4_overfitting_audit=pd.DataFrame(stage4_overfitting_audit)
stage4_train_validation_metrics.to_csv(STAGE4_EVIDENCE_DIR/'stage4_train_validation_metrics.csv',index=False)
stage4_overfitting_audit.to_csv(STAGE4_EVIDENCE_DIR/'stage4_overfitting_audit.csv',index=False)
display(stage4_train_validation_metrics); display(stage4_overfitting_audit)


## 4.7 Freeze Model Choices Before Test Access
Freeze each final model, its validation-selected threshold, and the primary model using validation PR-AUC. No test feature matrix or target is accessed in this section.


In [ ]:
# 4.7 Freeze all model and threshold choices using validation only.
stage4_validation_ranking=stage4_train_validation_metrics.query('split == "validation"').sort_values(['pr_auc','roc_auc'],ascending=[False,False],kind='mergesort').reset_index(drop=True)
primary_model_name=str(stage4_validation_ranking.iloc[0].model)
frozen_rows=[]
for name in selected_models:
    vr=stage4_train_validation_metrics.query('model == @name and split == "validation"').iloc[0]
    frozen_rows.append({'model':name,'candidate':selected_candidate_names[name],'threshold':selected_thresholds[name],'validation_pr_auc':vr.pr_auc,'validation_roc_auc':vr.roc_auc,'validation_f1':vr.f1,'primary_model':name==primary_model_name})
stage4_frozen_model_selection=pd.DataFrame(frozen_rows)
assert not any('test' in str(c).lower() for c in stage4_frozen_model_selection.columns)
STAGE4_SELECTION_FROZEN=True
PRIMARY_MODEL_THRESHOLD=float(selected_thresholds[primary_model_name])
assert 0.0 < PRIMARY_MODEL_THRESHOLD < 1.0
primary_validation_row=(stage4_train_validation_metrics.query("model == @primary_model_name and split == 'validation'").iloc[0])
PRIMARY_MODEL_VALIDATION_PR_AUC=float(primary_validation_row['pr_auc'])
PRIMARY_MODEL_VALIDATION_ROC_AUC=float(primary_validation_row['roc_auc'])
PRIMARY_MODEL_VALIDATION_F1=float(primary_validation_row['f1'])
stage4_frozen_model_selection.to_csv(STAGE4_EVIDENCE_DIR/'stage4_frozen_model_selection.csv',index=False)
print('STAGE4_SELECTION_FROZEN =',STAGE4_SELECTION_FROZEN)
print('Primary model selected BEFORE test access:',primary_model_name)
display(stage4_frozen_model_selection)



## 4.8 Unseen Test Evaluation
Now evaluate the untouched 2022–2024 test rows once, using the frozen model choices and validation thresholds. Test performance is descriptive only and cannot change the primary model.


In [ ]:
# 4.8 First Stage 4 test access occurs only after the selection-freeze flag.
assert STAGE4_SELECTION_FROZEN is True
X_test_s4=X_test_prepared
y_test_s4=np.asarray(y_test,dtype=int).ravel()
assert X_test_s4.shape[1]==feature_count_s4
assert X_test_s4.shape[0]==len(y_test_s4)
X_test_nn=np.asarray(X_test_s4,dtype=np.float32)
test_probs={}; test_rows=[]; conf_rows=[]; all_rows=[]
for name,model in selected_models.items():
    ptest=model.predict_proba(X_test_s4)[:,1] if hasattr(model,'predict_proba') else model.predict(X_test_nn,verbose=0).ravel(); test_probs[name]=ptest
    mt=stage4_metrics(y_train_s4,train_probs[name],selected_thresholds[name]); mv=stage4_metrics(y_val_s4,val_probs[name],selected_thresholds[name]); ms=stage4_metrics(y_test_s4,ptest,selected_thresholds[name])
    test_rows.append({'model':name,'candidate':selected_candidate_names[name],**ms})
    for split,met in [('train',mt),('validation',mv),('test',ms)]: all_rows.append({'model':name,'candidate':selected_candidate_names[name],'split':split,**met})
    tn,fp,fn,tp=confusion_matrix(y_test_s4,ptest>=selected_thresholds[name],labels=[0,1]).ravel()
    conf_rows.append({'model':name,'threshold':selected_thresholds[name],'TN':int(tn),'FP':int(fp),'FN':int(fn),'TP':int(tp),'test_rows':len(y_test_s4),'test_negatives':int((y_test_s4==0).sum()),'test_positives':int((y_test_s4==1).sum())})
stage4_test_metrics=pd.DataFrame(test_rows); stage4_all_split_model_comparison=pd.DataFrame(all_rows); stage4_test_confusion_matrices=pd.DataFrame(conf_rows)
display(stage4_test_metrics); display(stage4_test_confusion_matrices)


## 4.9 Model Comparison Plots
Generate validation and test ranking curves, thresholded metric comparisons, test confusion matrices, selected-network history, and threshold evidence. Plots are interpreted as predictive comparisons.


In [ ]:
# 4.9 Regenerate model-comparison tables, confusion evidence, and all Stage 4 plots.
stage4_test_metrics.to_csv(STAGE4_EVIDENCE_DIR/'stage4_test_metrics.csv',index=False)
stage4_all_split_model_comparison.to_csv(STAGE4_EVIDENCE_DIR/'stage4_all_split_model_comparison.csv',index=False)
stage4_test_confusion_matrices.to_csv(STAGE4_EVIDENCE_DIR/'stage4_test_confusion_matrices.csv',index=False)
stage4_train_validation_metrics.to_csv(STAGE4_EVIDENCE_DIR/'stage4_train_validation_metrics.csv',index=False)
colors={'Logistic Regression':'#1769aa','Random Forest':'#e87500','Shallow Neural Network':'#2a8c52'}
def stage4_savefig(fig,name):
    fig.tight_layout(); fig.savefig(STAGE4_EVIDENCE_DIR/name,dpi=150,bbox_inches='tight'); plt.show(); plt.close(fig)
for split,truth,probs in [('validation',y_val_s4,val_probs),('test',y_test_s4,test_probs)]:
    fig,ax=plt.subplots(figsize=(7,6)); ax.plot([0,1],[0,1],'--',color='gray',label='Chance')
    for name,p in probs.items():
        fpr,tpr,_=roc_curve(truth,p); ax.plot(fpr,tpr,color=colors[name],label=f"{name} (AUC={roc_auc_score(truth,p):.3f})")
    ax.set(xlabel='False positive rate',ylabel='True positive rate',title=f'{split.title()} ROC curves'); ax.legend(loc='lower right'); stage4_savefig(fig,f'stage4_{split}_roc.png')
    fig,ax=plt.subplots(figsize=(7,6)); ax.axhline(float(np.mean(truth)),ls='--',color='gray',label=f'No-skill prevalence={np.mean(truth):.3f}')
    for name,p in probs.items():
        pr,rc,_=precision_recall_curve(truth,p); ax.plot(rc,pr,color=colors[name],label=f"{name} (AP={average_precision_score(truth,p):.3f})")
    ax.set(xlabel='Recall',ylabel='Precision',title=f'{split.title()} Precision–Recall curves',ylim=(0,1.02)); ax.legend(); stage4_savefig(fig,f'stage4_{split}_pr.png')
fig,ax=plt.subplots(figsize=(9,5)); comp=stage4_all_split_model_comparison.query('split in ["validation","test"]')
for i,metric in enumerate(['roc_auc','pr_auc','f1']):
    for j,sp in enumerate(['validation','test']):
        vv=comp[comp.split.eq(sp)].set_index('model')[metric].reindex(list(selected_models)); x=np.arange(3)+(i-1)*.24+(j-.5)*.10
        ax.bar(x,vv.values,width=.10,label=f'{metric} / {sp}' if i==0 else None)
ax.set_xticks(np.arange(3),list(selected_models),rotation=10); ax.set_ylim(0,1); ax.set_ylabel('Score'); ax.set_title('Validation vs test model comparison'); ax.legend(ncol=2,fontsize=8); stage4_savefig(fig,'stage4_validation_test_metric_comparison.png')
for name in selected_models:
    row=stage4_test_confusion_matrices.query('model == @name').iloc[0]; mat=np.array([[row.TN,row.FP],[row.FN,row.TP]])
    fig,ax=plt.subplots(figsize=(5,4)); sns.heatmap(mat,annot=True,fmt='d',cmap='Blues',cbar=False,xticklabels=['Predicted 0','Predicted 1'],yticklabels=['Actual 0','Actual 1'],ax=ax)
    ax.set_title(f'Test confusion matrix — {name}'); stage4_savefig(fig,'stage4_test_confusion_'+name.lower().replace(' ','_')+'.png')
hist=nn_histories[selected_nn_name]
fig,ax=plt.subplots(figsize=(7,4)); ax.plot(hist['loss'],label='Training loss'); ax.plot(hist['val_loss'],label='Validation loss'); ax.set(xlabel='Epoch',ylabel='Binary cross-entropy',title='Selected NN loss; early stopping restored best weights'); ax.legend(); stage4_savefig(fig,'stage4_nn_training_loss.png')
fig,ax=plt.subplots(figsize=(7,4)); ax.plot(hist['pr_auc'],label='Training PR-AUC'); ax.plot(hist['val_pr_auc'],label='Validation PR-AUC'); ax.set(xlabel='Epoch',ylabel='PR-AUC',title='Selected NN PR-AUC training history'); ax.legend(); stage4_savefig(fig,'stage4_nn_training_pr_auc.png')
fig,ax=plt.subplots(figsize=(7,4)); ax.bar(stage4_threshold_selection.model,stage4_threshold_selection.default_f1,label='Default 0.50 F1'); ax.scatter(stage4_threshold_selection.model,stage4_threshold_selection.validation_f1,color='red',label='Selected threshold F1'); ax.set_ylim(0,1); ax.set_ylabel('Validation F1'); ax.set_title('Validation threshold selection'); ax.tick_params(axis='x',rotation=10); ax.legend(); stage4_savefig(fig,'stage4_threshold_selection.png')
display(stage4_all_split_model_comparison)


## 4.10 Final Model Choice & Limitations
Summarize the model selected from validation, report the frozen test result descriptively, and save fitted model/preprocessor artifacts. Predictive results do not establish causality.


In [ ]:
# 4.10 Save reusable fitted artifacts and write an evidence-based model interpretation.
params=[]
for name in selected_models:
    if name=='Logistic Regression': p={'C':float(logistic_selected_row.C),'class_weight':str(logistic_selected_row.class_weight),'solver':'liblinear'}
    elif name=='Random Forest': p={'n_estimators':400,'max_depth':None if pd.isna(rf_selected_row.max_depth) else int(rf_selected_row.max_depth),'min_samples_leaf':int(rf_selected_row.min_samples_leaf),'class_weight':str(rf_selected_row.class_weight)}
    else: p={'hidden_layers':[64,32] if selected_nn_name=='NN_3' else [64],
             'dropout':[.2,.1] if selected_nn_name=='NN_3' else [.2],'class_weight':str(nn_selected_row.class_weight),'epochs_trained':int(nn_selected_row.epochs_trained),'best_epoch':int(nn_selected_row.best_epoch)}
    params.append({'model':name,'selected_candidate':selected_candidate_names[name],'parameters':json.dumps(p),'threshold':selected_thresholds[name],'primary_model':name==primary_model_name,'transformed_features':feature_count_s4,'train_years':'1950-2017','validation_years':'2018-2021','test_years':'2022-2024'})
stage4_model_metadata=pd.DataFrame(params); stage4_model_metadata.to_csv(STAGE4_EVIDENCE_DIR/'stage4_model_metadata.csv',index=False)
joblib.dump(selected_logistic_model,STAGE4_EVIDENCE_DIR/'stage4_logistic_model.joblib'); joblib.dump(selected_rf_model,STAGE4_EVIDENCE_DIR/'stage4_random_forest_model.joblib'); joblib.dump(preprocessor,STAGE4_EVIDENCE_DIR/'stage4_preprocessor.joblib')
selected_nn_model.save(STAGE4_EVIDENCE_DIR/'stage4_shallow_nn.keras')
val_primary=stage4_train_validation_metrics.query('model == @primary_model_name and split == "validation"').iloc[0]; test_primary=stage4_test_metrics.query('model == @primary_model_name').iloc[0]
display(Markdown(f"Primary model: **{primary_model_name}**, selected before test access by validation PR-AUC {val_primary.pr_auc:.4f} (ROC-AUC {val_primary.roc_auc:.4f}); frozen threshold {selected_thresholds[primary_model_name]:.2f}. Descriptive test ROC-AUC/PR-AUC/F1 are {test_primary.roc_auc:.4f}/{test_primary.pr_auc:.4f}/{test_primary.f1:.4f}. Historical coverage and era shift limit generalization. Predictive importance or performance does not imply causation."))
display(stage4_model_metadata)

## 4.11 Stage 4 Validation
Validate frozen Stage 1–3 inputs, model candidates and selection, embargo order, metrics, confusion counts, and regenerated artifacts before continuing to experiments.


In [ ]:
# 4.11 Full Stage 4 validation and durable evidence summary.
checks4=[]
def add4(name,ok): checks4.append({'check':name,'passed':bool(ok)})
add4('Stage 1 validation passes',bool(validation_summary.passed.all()))
add4('Stage 2 remains 24/24 PASS',len(stage2_validation_summary)==24 and bool(stage2_validation_summary.passed.all()))
add4('Stage 3 remains 77/77 PASS',len(stage3_validation_summary)==77 and bool(stage3_validation_summary.passed.all()))
add4('Modeling rows remain 26,668',len(modeling_table)==26668)
add4('Train rows remain 23,689',len(y_train_s4)==23689)
add4('Validation rows remain 1,620',len(y_val_s4)==1620)
add4('Test rows remain 1,359',len(y_test_s4)==1359)
add4('Raw selected predictors remain 31',len(selected_predictors)==31)
add4('Transformed predictors remain 306',feature_count_s4==306)
add4('All splits have equal transformed feature counts',X_train_prepared.shape[1]==X_val_prepared.shape[1]==X_test_s4.shape[1]==306)
add4('Six logistic candidates trained',len(stage4_logistic_candidates)==6)
add4('Four RF candidates trained',len(stage4_random_forest_candidates)==4)
add4('Three shallow NN candidates trained',len(stage4_nn_candidates)==3)
add4('Unweighted C=1 logistic baseline retained',bool(((stage4_logistic_candidates.C==1)&(stage4_logistic_candidates.class_weight=='None')).any()))
add4('Selected models appear in their candidate tables',selected_logistic_name in set(stage4_logistic_candidates.candidate) and str(rf_selected_row.candidate) in set(stage4_random_forest_candidates.candidate) and selected_nn_name in set(stage4_nn_candidates.candidate))
add4('NN has only one or two hidden layers',all(1<=len(s['layers'])<=2 for s in nn_specs))
add4('NN class weights derive from training labels',set(train_class_weights)==set(np.unique(y_train_s4)))
add4('Primary model and thresholds frozen on validation before test',STAGE4_SELECTION_FROZEN and primary_model_name==str(stage4_validation_ranking.iloc[0].model) and not any('test' in str(c).lower() for c in stage4_frozen_model_selection.columns))
for split,probs in [('train',train_probs),('validation',val_probs),('test',test_probs)]:
    add4(f'{split.title()} probabilities finite and bounded',all(np.isfinite(p).all() and ((p>=0)&(p<=1)).all() for p in probs.values()))
for split in ['train','validation','test']:
    df=stage4_all_split_model_comparison.query('split == @split')
    for metric in ['roc_auc','pr_auc','f1','precision','recall','accuracy']:
        add4(f'{split.title()} {metric} is finite and within [0,1]',len(df)==3 and np.isfinite(df[metric]).all() and df[metric].between(0,1).all())
add4('Each test confusion matrix sums to 1,359',(stage4_test_confusion_matrices[['TN','FP','FN','TP']].sum(axis=1)==len(y_test_s4)).all())
add4('Test confusion negatives reconcile',(stage4_test_confusion_matrices.TN+stage4_test_confusion_matrices.FP==int((y_test_s4==0).sum())).all())
add4('Test confusion positives reconcile',(stage4_test_confusion_matrices.FN+stage4_test_confusion_matrices.TP==int((y_test_s4==1).sum())).all())
required_png4=['stage4_validation_roc.png','stage4_validation_pr.png','stage4_test_roc.png','stage4_test_pr.png','stage4_validation_test_metric_comparison.png','stage4_test_confusion_logistic_regression.png','stage4_test_confusion_random_forest.png','stage4_test_confusion_shallow_neural_network.png','stage4_nn_training_loss.png','stage4_nn_training_pr_auc.png','stage4_threshold_selection.png']
for fname in required_png4: add4(f'Plot exists: {fname}',(STAGE4_EVIDENCE_DIR/fname).is_file())
for fname,label in [('stage4_logistic_model.joblib','Logistic model artifact'),('stage4_random_forest_model.joblib','Random Forest artifact'),('stage4_shallow_nn.keras','Shallow NN artifact'),('stage4_preprocessor.joblib','Frozen preprocessor artifact')]: add4(label+' exists',(STAGE4_EVIDENCE_DIR/fname).is_file())
add4('Candidate CSVs exist',all((STAGE4_EVIDENCE_DIR/f).is_file() for f in ['stage4_logistic_candidates.csv','stage4_random_forest_candidates.csv','stage4_nn_candidates.csv']))
add4('Selection/threshold/comparison CSVs exist',all((STAGE4_EVIDENCE_DIR/f).is_file() for f in ['stage4_threshold_selection.csv','stage4_frozen_model_selection.csv','stage4_all_split_model_comparison.csv']))
add4('Final audits and metadata CSVs exist',all((STAGE4_EVIDENCE_DIR/f).is_file() for f in ['stage4_test_metrics.csv','stage4_test_confusion_matrices.csv','stage4_overfitting_audit.csv','stage4_model_metadata.csv']))
stage4_validation_summary=pd.DataFrame(checks4)
assert len(stage4_validation_summary)==60, f'Expected 60 Stage 4 checks, got {len(stage4_validation_summary)}.'
assert stage4_validation_summary.passed.all(),'Stage 4 validation failed.'
stage4_validation_summary.to_csv(STAGE4_EVIDENCE_DIR/'stage4_validation_summary.csv',index=False)
display(stage4_validation_summary)
print(f"PASS checks: {int(stage4_validation_summary.passed.sum())} / {len(stage4_validation_summary)}")
print('Stage 4 evidence files:',len(list(STAGE4_EVIDENCE_DIR.iterdir())))


# 5. Experiments
Experiments quantify feature-group contribution and demonstrate preprocessing-boundary effects. Stage 4 model selection remains frozen; Stage 5 uses training and validation only, with validation PR-AUC primary.

## 5.1 Experimental Protocol
Set seeds and assert the approved row sets and frozen Stage 4 primary model before controlled experiments.


In [ ]:
# 5.1 Frozen-model experiment setup; experiments below use TRAIN + VALIDATION only.
assert STAGE4_SELECTION_FROZEN is True and primary_model_name=='Shallow Neural Network'
RANDOM_STATE=42; np.random.seed(RANDOM_STATE); random.seed(RANDOM_STATE); tf.keras.utils.set_random_seed(RANDOM_STATE)
STAGE5_EVIDENCE_DIR=Path('/kaggle/working/stage5_experiment_evidence'); STAGE5_EVIDENCE_DIR.mkdir(parents=True,exist_ok=True)
STAGE5_TEST_USED=False
stage5_train_index=X_train_raw.index.copy(); stage5_val_index=X_val_raw.index.copy()
assert len(stage5_train_index)==23689 and len(stage5_val_index)==1620
print('Stage 5 protocol: frozen Stage 4 model choice; only train/validation rows; primary metric validation PR-AUC.')


## 5.2 Feature-Group Ablation
Remove one Stage 3 raw feature group at a time and retrain the frozen [64, 32] Shallow NN. Each preprocessor is fit on TRAIN only; validation controls early stopping and scoring. F1-based comparisons use the validation-selected frozen Stage 4 primary-model threshold, read directly from the Stage 4 selection output, and test data is excluded.

In [ ]:
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_score, recall_score, accuracy_score
import matplotlib.pyplot as plt

stage5_group_cols={g:list(cols) for g,cols in feature_groups.items()}
stage5_baseline={'pr_auc':PRIMARY_MODEL_VALIDATION_PR_AUC,'roc_auc':PRIMARY_MODEL_VALIDATION_ROC_AUC,'f1':PRIMARY_MODEL_VALIDATION_F1}
stage5_ablation_rows=[]
for group, removed in stage5_group_cols.items():
    keep=[c for c in selected_predictors if c not in removed]
    nums=[c for c in keep if c not in ['circuitId','constructorId']]
    cats=[c for c in keep if c in ['circuitId','constructorId']]
    try: enc=OneHotEncoder(handle_unknown='ignore',sparse_output=False)
    except TypeError: enc=OneHotEncoder(handle_unknown='ignore',sparse=False)
    numeric_pipe=Pipeline([('imputer',SimpleImputer(strategy='median',keep_empty_features=True)),('scaler',StandardScaler())])
    cat_pipe=Pipeline([('imputer',SimpleImputer(strategy='constant',fill_value='Missing')),('onehot',enc)])
    pp=ColumnTransformer([('numeric',numeric_pipe,nums),('categorical',cat_pipe,cats)],remainder='drop',verbose_feature_names_out=False)
    xt=np.asarray(pp.fit_transform(X_train_raw[keep]),dtype=np.float32); xv=np.asarray(pp.transform(X_val_raw[keep]),dtype=np.float32)
    tf.keras.backend.clear_session(); tf.keras.utils.set_random_seed(RANDOM_STATE)
    inp=tf.keras.Input(shape=(xt.shape[1],)); z=tf.keras.layers.Dense(64,activation='relu')(inp); z=tf.keras.layers.Dropout(.20)(z); z=tf.keras.layers.Dense(32,activation='relu')(z); z=tf.keras.layers.Dropout(.10)(z); out=tf.keras.layers.Dense(1,activation='sigmoid')(z); model=tf.keras.Model(inp,out)
    model.compile(optimizer=tf.keras.optimizers.Adam(),loss='binary_crossentropy',metrics=[tf.keras.metrics.AUC(curve='PR',name='pr_auc')])
    cw=compute_class_weight(class_weight='balanced',classes=np.unique(y_train_s4),y=y_train_s4); class_weights={int(c):float(w) for c,w in zip(np.unique(y_train_s4),cw)}
    cb=tf.keras.callbacks.EarlyStopping(monitor='val_pr_auc',mode='max',patience=8,restore_best_weights=True)
    hist=model.fit(xt,y_train_s4,validation_data=(xv,y_val_s4),epochs=100,batch_size=128,verbose=0,callbacks=[cb],class_weight=class_weights)
    p=model.predict(xv,verbose=0).ravel(); pred=(p>=PRIMARY_MODEL_THRESHOLD).astype(int)
    row={'removed_group':group,'removed_features':len(removed),'remaining_raw_features':len(keep),'transformed_features':int(xt.shape[1]),'epochs':len(hist.history['loss']),'roc_auc':roc_auc_score(y_val_s4,p),'pr_auc':average_precision_score(y_val_s4,p),'frozen_threshold':PRIMARY_MODEL_THRESHOLD,'f1_at_frozen_threshold':f1_score(y_val_s4,pred,zero_division=0),'precision_at_frozen_threshold':precision_score(y_val_s4,pred,zero_division=0),'recall_at_frozen_threshold':recall_score(y_val_s4,pred,zero_division=0),'accuracy_at_frozen_threshold':accuracy_score(y_val_s4,pred)}
    row.update({'delta_pr_auc':stage5_baseline['pr_auc']-row['pr_auc'],'delta_roc_auc':stage5_baseline['roc_auc']-row['roc_auc'],'delta_f1':stage5_baseline['f1']-row['f1_at_frozen_threshold']}); stage5_ablation_rows.append(row); del model,pp,xt,xv
stage5_ablation=pd.DataFrame(stage5_ablation_rows).sort_values('delta_pr_auc',ascending=False)
stage5_ablation.to_csv(STAGE5_EVIDENCE_DIR/'stage5_feature_group_ablation.csv',index=False)
fig,ax=plt.subplots(figsize=(8,4.5)); q=stage5_ablation.sort_values('delta_pr_auc'); ax.barh(q.removed_group,q.delta_pr_auc,color='#2878B5'); ax.axvline(0,color='black',lw=.8); ax.set_xlabel('Frozen baseline PR-AUC − ablated validation PR-AUC'); ax.set_title('Feature-group ablation (validation PR-AUC drop)'); fig.tight_layout(); fig.savefig(STAGE5_EVIDENCE_DIR/'stage5_feature_group_ablation_pr_auc.png',dpi=160); plt.show(); display(stage5_ablation)
stage5_ablation_plot=STAGE5_EVIDENCE_DIR/'stage5_feature_group_ablation_pr_auc.png'
stage5_most_group=str(stage5_ablation.iloc[0].removed_group); stage5_least_group=str(stage5_ablation.iloc[-1].removed_group)
print('Frozen full-model validation PR-AUC baseline:',stage5_baseline['pr_auc'],'; threshold:',PRIMARY_MODEL_THRESHOLD,'; largest drop:',stage5_most_group,'; smallest:',stage5_least_group)

## 5.3 Preprocessing-Order Experiment 1 ? Split Before Preprocessing vs Before Split
This controlled TRAIN + VALIDATION-only demonstration contrasts the approved train-fitted pipeline with an explicitly invalid/leaky demonstration. Both use the selected Logistic Regression configuration and its Stage 4 validation-selected threshold, retrieved from the frozen Stage 4 threshold record; neither result changes production selection.

In [ ]:
# Approved: frozen Stage 3 train-fitted preprocessing and chronological split.
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score,average_precision_score,f1_score,precision_score,recall_score,accuracy_score
from IPython.display import display,Markdown

stage5_lr_threshold=float(stage4_threshold_selection.loc[stage4_threshold_selection.model.eq('Logistic Regression'),'threshold'].iloc[0])
assert np.isclose(stage5_lr_threshold, float(selected_thresholds['Logistic Regression']))
def stage5_lr_metrics(y,p,threshold):
    pred=(p>=threshold).astype(int)
    return {'roc_auc':roc_auc_score(y,p),'pr_auc':average_precision_score(y,p),'f1':f1_score(y,pred,zero_division=0),'precision':precision_score(y,pred,zero_division=0),'recall':recall_score(y,pred,zero_division=0),'accuracy':accuracy_score(y,pred)}
model_ok=LogisticRegression(C=.1,class_weight=None,solver='liblinear',random_state=RANDOM_STATE,max_iter=1000)
model_ok.fit(X_train_prepared,y_train_s4); p_ok=model_ok.predict_proba(X_val_prepared)[:,1]
row_ok={'pipeline':'APPROVED — split first; preprocess fit on TRAIN only','threshold':stage5_lr_threshold,**stage5_lr_metrics(y_val_s4,p_ok,stage5_lr_threshold)}
# Intentionally invalid demonstration: preprocess on combined TRAIN + VALIDATION raw rows only.
raw_tv=pd.concat([X_train_raw,X_val_raw],axis=0); keep_cols=list(selected_predictors); nums=[c for c in keep_cols if c not in ['circuitId','constructorId']]; cats=[c for c in keep_cols if c in ['circuitId','constructorId']]
try: enc=OneHotEncoder(handle_unknown='ignore',sparse_output=False)
except TypeError: enc=OneHotEncoder(handle_unknown='ignore',sparse=False)
leaky_pp=ColumnTransformer([('numeric',Pipeline([('imputer',SimpleImputer(strategy='median',keep_empty_features=True)),('scaler',StandardScaler())]),nums),('categorical',Pipeline([('imputer',SimpleImputer(strategy='constant',fill_value='Missing')),('onehot',enc)]),cats)],remainder='drop',verbose_feature_names_out=False)
Ztv=leaky_pp.fit_transform(raw_tv); Ztr=Ztv[:len(X_train_raw)]; Zva=Ztv[len(X_train_raw):]
model_leaky=LogisticRegression(C=.1,class_weight=None,solver='liblinear',random_state=RANDOM_STATE,max_iter=1000); model_leaky.fit(Ztr,y_train_s4); p_leaky=model_leaky.predict_proba(Zva)[:,1]
row_leaky={'pipeline':'INVALID / LEAKY DEMONSTRATION — NOT USED FOR MODEL SELECTION','threshold':stage5_lr_threshold,**stage5_lr_metrics(y_val_s4,p_leaky,stage5_lr_threshold)}
stage5_preprocessing_order_1=pd.DataFrame([row_ok,row_leaky]); stage5_preprocessing_order_1.to_csv(STAGE5_EVIDENCE_DIR/'stage5_preprocessing_order_experiment_1.csv',index=False)
fig,ax=plt.subplots(figsize=(8,4)); xx=np.arange(2); ax.bar(xx-.18,stage5_preprocessing_order_1.pr_auc,.36,label='PR-AUC'); ax.bar(xx+.18,stage5_preprocessing_order_1.roc_auc,.36,label='ROC-AUC'); ax.set_xticks(xx,['Approved split-first','Invalid leaky demo']); ax.set_ylim(0,1); ax.set_ylabel('Validation score'); ax.set_title('Preprocessing boundary demonstration'); ax.legend(); fig.tight_layout(); fig.savefig(STAGE5_EVIDENCE_DIR/'stage5_preprocessing_order_experiment_1.png',dpi=160); plt.show(); display(stage5_preprocessing_order_1)
display(Markdown('Even a small metric difference does not make preprocessing before the chronological split valid: validation distribution information influenced fitted preprocessing. This demonstration is excluded from selection.'))

## 5.4 Preprocessing-Order Experiment 2 ? Numeric Imputation and Scaling
Compare median imputation followed by standard scaling with NaN-tolerant scaling followed by median imputation. Categorical processing stays fixed, both pipelines fit on TRAIN only, and validation is scored at the same Stage 4 validation-selected Logistic Regression threshold used in Experiment 1.

In [ ]:
stage5_order2_rows=[]
stage5_lr_threshold=float(stage4_threshold_selection.loc[stage4_threshold_selection.model.eq('Logistic Regression'),'threshold'].iloc[0])
assert np.isclose(stage5_lr_threshold, float(selected_thresholds['Logistic Regression']))
# A: approved median imputation -> scaling.
num_a=Pipeline([('imputer',SimpleImputer(strategy='median',keep_empty_features=True)),('scaler',StandardScaler())])
# B: StandardScaler preserves NaNs in supported sklearn versions; median imputation follows it.
num_b=Pipeline([('scaler',StandardScaler()),('imputer',SimpleImputer(strategy='median',keep_empty_features=True))])
for label,num_pipe in [('APPROVED: median imputation -> StandardScaler',num_a),('ALTERNATIVE: StandardScaler -> median imputation',num_b)]:
    try: enc=OneHotEncoder(handle_unknown='ignore',sparse_output=False)
    except TypeError: enc=OneHotEncoder(handle_unknown='ignore',sparse=False)
    pp=ColumnTransformer([('numeric',num_pipe,numeric_features),('categorical',Pipeline([('imputer',SimpleImputer(strategy='constant',fill_value='Missing')),('onehot',enc)]),categorical_features)],remainder='drop',verbose_feature_names_out=False)
    ztr=np.asarray(pp.fit_transform(X_train_raw),dtype=float); zva=np.asarray(pp.transform(X_val_raw),dtype=float)
    lr=LogisticRegression(C=.1,class_weight=None,solver='liblinear',random_state=RANDOM_STATE,max_iter=1000); lr.fit(ztr,y_train_s4); ppred=lr.predict_proba(zva)[:,1]
    stage5_order2_rows.append({'pipeline':label,'transformers_fit_on':'TRAIN only','threshold':stage5_lr_threshold,**stage5_lr_metrics(y_val_s4,ppred,stage5_lr_threshold)})
stage5_preprocessing_order_2=pd.DataFrame(stage5_order2_rows); stage5_preprocessing_order_2.to_csv(STAGE5_EVIDENCE_DIR/'stage5_preprocessing_order_experiment_2.csv',index=False)
fig,ax=plt.subplots(figsize=(8,4)); x=np.arange(len(stage5_preprocessing_order_2)); w=.25
for i,m in enumerate(['roc_auc','pr_auc','f1']): ax.bar(x+(i-1)*w,stage5_preprocessing_order_2[m],w,label=m.upper())
ax.set_xticks(x,['Impute → scale','Scale → impute']); ax.set_ylim(0,1); ax.set_ylabel('Validation score'); ax.set_title('Numeric preprocessing order (train-fitted)'); ax.legend(); fig.tight_layout(); fig.savefig(STAGE5_EVIDENCE_DIR/'stage5_preprocessing_order_experiment_2.png',dpi=160); plt.show(); display(stage5_preprocessing_order_2)
display(Markdown('The two train-fitted orders are compared on the same chronological validation rows with identical logistic settings. The approved production pipeline remains median imputation followed by scaling regardless of the observed score difference.'))

## 5.5 Experiment Findings
Summarize the observed ablation and preprocessing results. Deltas are descriptive validation comparisons; they do not establish causal effects or change the frozen Stage 4 model.

In [ ]:
stage5_most=stage5_ablation.iloc[0]; stage5_least=stage5_ablation.iloc[-1]
stage5_experiment_findings=(f"Removing **{stage5_most.removed_group}** had the largest observed validation PR-AUC drop ({stage5_most.delta_pr_auc:.4f}); removing **{stage5_least.removed_group}** had the smallest ({stage5_least.delta_pr_auc:.4f}). Feature overlap can make incremental drops small. The split-order experiment labels its combined-fit pipeline invalid; its score is not evidence of valid generalization. The numeric-order results compare two train-fitted alternatives. These are predictive associations, not causal effects.")
display(Markdown(stage5_experiment_findings))

## 5.6 Stage 5 Validation
Validate the six group ablations, frozen thresholds, train-only preprocessing, and saved experiment evidence. This audit explicitly confirms test data was not used by Stage 5.

In [ ]:
stage5_checks=[
('six feature groups ablated',len(stage5_ablation)==6),
('all six approved groups represented',set(stage5_ablation.removed_group)==set(stage5_group_cols)),
('baseline equals frozen Stage 4 validation PR-AUC',np.isclose(stage5_baseline['pr_auc'],PRIMARY_MODEL_VALIDATION_PR_AUC)),
('all ablations use frozen Stage 4 threshold',np.allclose(stage5_ablation.frozen_threshold,PRIMARY_MODEL_THRESHOLD) and np.isclose(PRIMARY_MODEL_THRESHOLD,selected_thresholds[primary_model_name])),
('ablation preprocessors fit on train only',len(stage5_train_index)==23689),
('train/validation row sets unchanged',stage5_train_index.equals(X_train_raw.index) and stage5_val_index.equals(X_val_raw.index)),
('test data not used in Stage 5',STAGE5_TEST_USED is False),
('split-order experiment has approved and labeled leaky rows',len(stage5_preprocessing_order_1)==2 and 'INVALID / LEAKY DEMONSTRATION' in stage5_preprocessing_order_1.pipeline.iloc[1]),
('order experiment 1 evidence saved',(STAGE5_EVIDENCE_DIR/'stage5_preprocessing_order_experiment_1.csv').exists()),
('two train-only numeric orders compared',len(stage5_preprocessing_order_2)==2 and (stage5_preprocessing_order_2.transformers_fit_on=='TRAIN only').all()),
('ablation CSV and plot saved',(STAGE5_EVIDENCE_DIR/'stage5_feature_group_ablation.csv').exists() and stage5_ablation_plot.exists()),
('preprocessing comparison plots saved',all((STAGE5_EVIDENCE_DIR/f'stage5_preprocessing_order_experiment_{i}.png').exists() for i in [1,2])),
('all ablation metrics finite',np.isfinite(stage5_ablation[['roc_auc','pr_auc','f1_at_frozen_threshold','precision_at_frozen_threshold','recall_at_frozen_threshold','accuracy_at_frozen_threshold']].to_numpy()).all()),
('non-causal interpretation present','not causal' in stage5_experiment_findings.lower())]
stage5_validation_summary=pd.DataFrame(stage5_checks,columns=['check','passed']); stage5_validation_summary.to_csv(STAGE5_EVIDENCE_DIR/'stage5_validation_summary.csv',index=False); display(stage5_validation_summary); print(f"PASS checks: {int(stage5_validation_summary.passed.sum())} / {len(stage5_validation_summary)}"); assert stage5_validation_summary.passed.all()

# 6. Model Explainability (XAI)
This stage uses the frozen Stage 4 primary Shallow NN. Global and local explanations describe model behavior on held-out validation records; they are not causal effects.

## 6.1 XAI Protocol & Frozen Inputs
Confirm the selected model and validation-only explanation boundary, then create an evidence folder and deterministic seed.

In [ ]:
assert selected_nn_name == 'NN_3'
assert STAGE4_SELECTION_FROZEN is True
assert primary_model_name == 'Shallow Neural Network'
assert np.isclose(PRIMARY_MODEL_THRESHOLD, selected_thresholds[primary_model_name])
assert 0 < PRIMARY_MODEL_THRESHOLD < 1
STAGE6_EVIDENCE_DIR=Path('/kaggle/working/stage6_xai_evidence'); STAGE6_EVIDENCE_DIR.mkdir(parents=True,exist_ok=True)
np.random.seed(RANDOM_STATE); random.seed(RANDOM_STATE); tf.keras.utils.set_random_seed(RANDOM_STATE)
STAGE6_TEST_USED=False
print('XAI primary model:',primary_model_name,'; validation rows:',len(X_val_raw),'; raw features:',len(selected_predictors))

## 6.2 Global Raw-Feature Permutation Importance
For each of the 31 raw predictors, shuffle one validation column five times, transform through the frozen train-fitted preprocessor, and measure the primary NN's validation PR-AUC drop. This reports reliance under permutation, not causation.

In [ ]:
from sklearn.metrics import average_precision_score
permutation_baseline=float(average_precision_score(y_val_s4,selected_nn_model.predict(np.asarray(X_val_prepared,dtype=np.float32),verbose=0).ravel()))
perm_rng=np.random.default_rng(RANDOM_STATE); permutation_rows=[]
for feature in selected_predictors:
    drops=[]
    for rep in range(5):
        raw=X_val_raw.copy(); raw[feature]=perm_rng.permutation(raw[feature].to_numpy())
        z=np.asarray(preprocessor.transform(raw),dtype=np.float32); p=selected_nn_model.predict(z,verbose=0).ravel()
        drops.append(permutation_baseline-average_precision_score(y_val_s4,p))
    grp=next((g for g,cols in feature_groups.items() if feature in cols),'unassigned')
    permutation_rows.append({'feature':feature,'group':grp,'baseline_validation_pr_auc':permutation_baseline,'mean_permuted_pr_auc':permutation_baseline-float(np.mean(drops)),'importance_mean_pr_auc_drop':float(np.mean(drops)),'importance_std':float(np.std(drops,ddof=1))})
stage6_permutation_importance=pd.DataFrame(permutation_rows).sort_values('importance_mean_pr_auc_drop',ascending=False)
stage6_permutation_importance.to_csv(STAGE6_EVIDENCE_DIR/'stage6_permutation_importance.csv',index=False)
fig,ax=plt.subplots(figsize=(9,6)); q=stage6_permutation_importance.head(15).sort_values('importance_mean_pr_auc_drop'); ax.barh(q.feature,q.importance_mean_pr_auc_drop,xerr=q.importance_std,color='#2878B5'); ax.set_xlabel('Validation PR-AUC drop after shuffling'); ax.set_title('Global raw-feature permutation importance (15 highest)'); fig.tight_layout(); fig.savefig(STAGE6_EVIDENCE_DIR/'stage6_permutation_importance.png',dpi=160); plt.show(); display(stage6_permutation_importance.head(10))
stage6_top_permutation=stage6_permutation_importance.head(10).feature.tolist()
display(Markdown('Features with larger drops were more relied on by this fitted model on these validation rows. Correlated predictors can share or mask importance; this ranking is not causal.'))

## 6.3 Global SHAP Explanation
Estimate global SHAP values for the frozen primary NN using training background rows and validation explanation rows. If the NN-specific SHAP explainer is incompatible, try a compact Kernel SHAP explanation; use a clearly labeled Random Forest Tree SHAP fallback only if both NN methods fail.

In [ ]:
import shap, warnings
stage6_shap_method=''; stage6_shap_values=None; stage6_shap_X=None
shap_background=np.asarray(X_train_prepared,dtype=np.float32)[np.random.default_rng(RANDOM_STATE).choice(len(X_train_prepared),size=min(100,len(X_train_prepared)),replace=False)]
shap_eval=np.asarray(X_val_prepared,dtype=np.float32)[:min(100,len(X_val_prepared))]
def shap_nn_predict(z): return selected_nn_model.predict(np.asarray(z,dtype=np.float32),verbose=0).ravel()
try:
    ex=shap.GradientExplainer(selected_nn_model,shap_background); sv=ex.shap_values(shap_eval)
    if isinstance(sv,list): sv=sv[-1]
    sv=np.asarray(sv)
    if sv.ndim==3: sv=sv[...,0] if sv.shape[-1]==1 else sv[:,:,-1]
    if sv.shape!=shap_eval.shape: raise ValueError(f'Unexpected Gradient SHAP shape {sv.shape}')
    stage6_shap_values=sv; stage6_shap_method='GradientExplainer — primary Shallow NN'
except Exception as grad_error:
    stage6_gradient_error=repr(grad_error)[:350]
    try:
        bg=shap_background[:min(10,len(shap_background))]; ev=shap_eval[:min(20,len(shap_eval))]
        ex=shap.KernelExplainer(shap_nn_predict,bg); sv=ex.shap_values(ev,nsamples=50)
        if isinstance(sv,list): sv=sv[-1]
        sv=np.asarray(sv)
        if sv.ndim==3: sv=sv[...,0] if sv.shape[-1]==1 else sv[:,:,-1]
        if sv.shape!=ev.shape: raise ValueError(f'Unexpected Kernel SHAP shape {sv.shape}')
        shap_eval=ev; stage6_shap_values=sv; stage6_shap_method='KernelExplainer — primary Shallow NN (compact fallback)'
    except Exception as kernel_error:
        stage6_kernel_error=repr(kernel_error)[:350]
        ex=shap.TreeExplainer(selected_rf_model); sv=ex.shap_values(np.asarray(X_val_prepared,dtype=np.float32)[:min(100,len(X_val_prepared))])
        if isinstance(sv,list): sv=sv[-1]
        sv=np.asarray(sv)
        if sv.ndim==3: sv=sv[:,:,-1]
        shap_eval=np.asarray(X_val_prepared,dtype=np.float32)[:len(sv)]; stage6_shap_values=sv; stage6_shap_method='TreeExplainer — Random Forest fallback (NN remains primary)'
stage6_shap_X=pd.DataFrame(shap_eval,columns=transformed_feature_names)
stage6_shap_importance=pd.DataFrame({'transformed_feature':transformed_feature_names,'mean_abs_shap':np.abs(stage6_shap_values).mean(axis=0)}).sort_values('mean_abs_shap',ascending=False)
stage6_shap_importance.to_csv(STAGE6_EVIDENCE_DIR/'stage6_shap_global_importance.csv',index=False)
fig=plt.figure(figsize=(10,6)); shap.summary_plot(stage6_shap_values,stage6_shap_X,show=False,max_display=15); plt.title(f'Global SHAP — {stage6_shap_method}'); plt.tight_layout(); plt.savefig(STAGE6_EVIDENCE_DIR/'stage6_shap_beeswarm.png',dpi=160,bbox_inches='tight'); plt.show()
fig=plt.figure(figsize=(8,5)); shap.summary_plot(stage6_shap_values,stage6_shap_X,plot_type='bar',show=False,max_display=15); plt.title('Global mean absolute SHAP'); plt.tight_layout(); plt.savefig(STAGE6_EVIDENCE_DIR/'stage6_shap_bar.png',dpi=160,bbox_inches='tight'); plt.show()
stage6_top_shap=stage6_shap_importance.head(10).transformed_feature.tolist(); display(stage6_shap_importance.head(10)); print('SHAP method used:',stage6_shap_method)
display(Markdown('SHAP summarizes how transformed predictors move model output on the sampled validation rows relative to the chosen background. Correlated features and era-specific data limit interpretation; attribution is not causal.'))

## 6.4 Local LIME Explanation
Select a reproducible validation record, prefer a podium row when available, and explain the frozen NN score with LIME over transformed features. The explanation is local and approximate; no post-race outcome is an input.

In [ ]:
try:
    from lime.lime_tabular import LimeTabularExplainer
except ImportError:
    import subprocess,sys
    subprocess.check_call([sys.executable,'-m','pip','install','lime','-q'])
    from lime.lime_tabular import LimeTabularExplainer
val_nn_probs=selected_nn_model.predict(np.asarray(X_val_prepared,dtype=np.float32),verbose=0).ravel()
pos_rows=np.flatnonzero(y_val_s4==1); stage6_local_idx=int(pos_rows[0] if len(pos_rows) else np.argmax(val_nn_probs))
X_train_lime=np.asarray(X_train_prepared,dtype=float); X_val_lime=np.asarray(X_val_prepared,dtype=float)
def lime_predict_proba(z):
    p=selected_nn_model.predict(np.asarray(z,dtype=np.float32),verbose=0).ravel(); return np.column_stack([1-p,p])
lime_exp=LimeTabularExplainer(X_train_lime[np.random.default_rng(RANDOM_STATE).choice(len(X_train_lime),size=min(1500,len(X_train_lime)),replace=False)],feature_names=transformed_feature_names,class_names=['No podium','Podium'],mode='classification',discretize_continuous=True,random_state=RANDOM_STATE)
stage6_lime_explanation=lime_exp.explain_instance(X_val_lime[stage6_local_idx],lime_predict_proba,num_features=15,labels=(1,),num_samples=2500)
stage6_lime_rows=[{'feature_condition':f,'local_weight':float(w),'validation_row_index':int(X_val_raw.index[stage6_local_idx])} for f,w in stage6_lime_explanation.as_list(label=1)]
stage6_lime_table=pd.DataFrame(stage6_lime_rows).sort_values('local_weight',ascending=False); stage6_lime_table.to_csv(STAGE6_EVIDENCE_DIR/'stage6_lime_local_explanation.csv',index=False)
fig=stage6_lime_explanation.as_pyplot_figure(label=1); fig.set_size_inches(9,5); fig.tight_layout(); fig.savefig(STAGE6_EVIDENCE_DIR/'stage6_lime_local.png',dpi=160,bbox_inches='tight'); plt.show()
stage6_lime_probability=float(val_nn_probs[stage6_local_idx]); stage6_lime_prediction=int(stage6_lime_probability>=PRIMARY_MODEL_THRESHOLD); stage6_lime_strong_positive=stage6_lime_table.head(5).feature_condition.tolist(); stage6_lime_strong_negative=stage6_lime_table.tail(5).sort_values('local_weight').feature_condition.tolist()
print('Explained validation row:',int(X_val_raw.index[stage6_local_idx]),'actual podium:',int(y_val_s4[stage6_local_idx]),'probability:',round(stage6_lime_probability,4),'threshold:',PRIMARY_MODEL_THRESHOLD,'predicted class:',stage6_lime_prediction); display(stage6_lime_table)

## 6.5 Stage 6 Interpretation & Validation
Record the strongest global and local explanation results and validate the validation-only boundary, model identity, and generated XAI evidence. Importance describes this fitted model, not causal influence.

In [ ]:
stage6_xai_interpretation=(f"Permutation ranking's top raw feature was **{stage6_top_permutation[0]}**. Top SHAP attributions included **{', '.join(stage6_top_shap[:5])}** using {stage6_shap_method}. For the selected local validation row, predicted podium probability was {stage6_lime_probability:.3f} (threshold {PRIMARY_MODEL_THRESHOLD:.2f}); strongest positive local conditions were {', '.join(stage6_lime_strong_positive[:3])}, while strongest negative conditions were {', '.join(stage6_lime_strong_negative[:3])}. These explanations are model attributions and do not establish causality.")
display(Markdown(stage6_xai_interpretation))
stage6_checks=[('frozen Stage 4 primary remains Shallow NN',primary_model_name=='Shallow Neural Network' and STAGE4_SELECTION_FROZEN),('permutation importance covers 31 raw features',len(stage6_permutation_importance)==31 and set(stage6_permutation_importance.feature)==set(selected_predictors)),('permutation importance uses validation rows',len(y_val_s4)==1620),('permutation importance plot exists',(STAGE6_EVIDENCE_DIR/'stage6_permutation_importance.png').exists()),('SHAP method completed',isinstance(stage6_shap_method,str) and len(stage6_shap_method)>0),('SHAP plot outputs exist',(STAGE6_EVIDENCE_DIR/'stage6_shap_beeswarm.png').exists() and (STAGE6_EVIDENCE_DIR/'stage6_shap_bar.png').exists()),('LIME local explanation completed',len(stage6_lime_table)>0),('LIME probability valid',0<=stage6_lime_probability<=1),('LIME plot and CSV exist',(STAGE6_EVIDENCE_DIR/'stage6_lime_local.png').exists() and (STAGE6_EVIDENCE_DIR/'stage6_lime_local_explanation.csv').exists()),('Stage 6 did not use test',STAGE6_TEST_USED is False),('causal disclaimer included','not establish causality' in stage6_xai_interpretation.lower())]
stage6_validation_summary=pd.DataFrame(stage6_checks,columns=['check','passed']); stage6_validation_summary.to_csv(STAGE6_EVIDENCE_DIR/'stage6_validation_summary.csv',index=False); display(stage6_validation_summary); print(f"PASS checks: {int(stage6_validation_summary.passed.sum())} / {len(stage6_validation_summary)}"); assert stage6_validation_summary.passed.all()

# 7. Final Inference Function
Expose a small pre-race scoring interface backed by the frozen train-fitted preprocessor, selected Shallow NN, and fixed validation threshold. It requires only the 31 approved predictors and accepts missing values handled by the fitted preprocessing pipeline.

## 7.1 Inference Function


In [ ]:
def predict_podium_companion(record):
    """Score one pre-race record with frozen Stage 4 artifacts; accepts dict/Series/one-row DataFrame."""
    if isinstance(record,pd.DataFrame):
        if len(record)!=1: raise ValueError('Provide exactly one inference record.')
        row=record.iloc[0]
    elif isinstance(record,pd.Series): row=record
    elif isinstance(record,dict): row=pd.Series(record)
    else: raise TypeError('record must be a dict, Series, or one-row DataFrame')
    raw=pd.DataFrame([row.reindex(selected_predictors).to_dict()],columns=selected_predictors)
    for c in numeric_features: raw[c]=pd.to_numeric(raw[c],errors='coerce').replace([np.inf,-np.inf],np.nan)
    missing_predictors=int(raw.isna().sum(axis=1).iloc[0])
    provided_predictors=len(selected_predictors)-missing_predictors
    for c in categorical_features: raw[c]=raw[c].astype('string').fillna('Missing').astype(str)
    transformed=np.asarray(preprocessor.transform(raw),dtype=np.float32)
    probability=float(selected_nn_model.predict(transformed,verbose=0).ravel()[0]); threshold=PRIMARY_MODEL_THRESHOLD
    return {'model':primary_model_name,'probability':probability,'threshold':threshold,'predicted_class':int(probability>=threshold),'provided_predictors':provided_predictors,'missing_predictors':missing_predictors,'predictor_count':len(selected_predictors)}
assert STAGE4_SELECTION_FROZEN
assert np.isclose(PRIMARY_MODEL_THRESHOLD, selected_thresholds[primary_model_name])
print('Inference schema:',len(selected_predictors),'predictors; model:',primary_model_name,'; threshold:',selected_thresholds[primary_model_name])

## 7.2 Inference Demo — Complete Pre-Race Record
Use the validation-era raw predictor record with the fewest missing predictors. The demonstration row includes only approved predictors; race outcomes are neither required nor passed to the inference function.

In [ ]:
provided=X_val_raw.notna().sum(axis=1); stage7_complete_index=provided.idxmax(); stage7_complete_record=X_val_raw.loc[stage7_complete_index].copy(); stage7_complete_result=predict_podium_companion(stage7_complete_record)
stage7_complete_output=pd.DataFrame([stage7_complete_result]); stage7_complete_output.insert(0,'source_row_index',int(stage7_complete_index)); stage7_complete_output.to_csv('/kaggle/working/stage7_inference_complete_record.csv',index=False)
display(stage7_complete_output); print('Complete-record prediction generated from',stage7_complete_result['provided_predictors'],'provided predictors.')

## 7.3 Inference Demo — Partially Missing Record
Starting from the same real pre-race row, intentionally omit qualifying gaps and previous-standings predictors that may legitimately be unavailable. The fitted training imputers and indicators allow a score without removing all signal.

In [ ]:
stage7_partial_record=stage7_complete_record.copy()
for c in ['q2_gap_to_best','q3_gap_to_best','driver_prev_standing_position','constructor_prev_points_vs_leader']:
    if c in stage7_partial_record.index: stage7_partial_record[c]=np.nan
stage7_partial_result=predict_podium_companion(stage7_partial_record)
stage7_partial_output=pd.DataFrame([stage7_partial_result]); stage7_partial_output.insert(0,'source_row_index',int(stage7_complete_index)); stage7_partial_output.to_csv('/kaggle/working/stage7_inference_partial_record.csv',index=False)
display(stage7_partial_output); print('Partial-record prediction accepted',stage7_partial_result['missing_predictors'],'missing predictors.')

## 7.4 Inference Validation
Check the predictor schema, frozen model and threshold, output range, missing-value behavior, and generated demonstration evidence.

In [ ]:
fig,ax=plt.subplots(figsize=(6,4)); vals=[stage7_complete_result['probability'],stage7_partial_result['probability']]; ax.bar(['Complete record','Partially missing'],vals,color=['#2878B5','#F0A202']); ax.axhline(PRIMARY_MODEL_THRESHOLD,color='red',ls='--',label=f'Frozen threshold {PRIMARY_MODEL_THRESHOLD:.2f}'); ax.set_ylim(0,1); ax.set_ylabel('Podium probability'); ax.set_title('Inference demonstrations'); ax.legend(); fig.tight_layout(); fig.savefig('/kaggle/working/stage7_inference_demonstrations.png',dpi=160); plt.show()
stage7_checks=[('inference function defined',callable(predict_podium_companion)),('same 31 predictor schema',len(selected_predictors)==31 and stage7_complete_result['predictor_count']==31),('frozen Stage 3 preprocessor used',preprocessor is not None),('frozen primary Shallow NN used',primary_model_name=='Shallow Neural Network'),('threshold matches frozen Stage 4 selection',np.isclose(stage7_complete_result['threshold'],PRIMARY_MODEL_THRESHOLD) and np.isclose(stage7_partial_result['threshold'],PRIMARY_MODEL_THRESHOLD) and np.isclose(PRIMARY_MODEL_THRESHOLD,selected_thresholds[primary_model_name])),('complete probability in [0,1]',0<=stage7_complete_result['probability']<=1),('partial probability in [0,1]',0<=stage7_partial_result['probability']<=1),('function accepts legitimate missing values',stage7_partial_result['missing_predictors']>=4),('post-race outcome fields not required',not any(c in selected_predictors for c in ['podium','positionOrder','points','statusId'])),('output schema correct',set(['model','probability','threshold','predicted_class','provided_predictors','missing_predictors','predictor_count']).issubset(stage7_complete_result)),('inference CSVs regenerated',__import__('pathlib').Path('/kaggle/working/stage7_inference_complete_record.csv').exists() and __import__('pathlib').Path('/kaggle/working/stage7_inference_partial_record.csv').exists()),('inference summary figure regenerated',__import__('pathlib').Path('/kaggle/working/stage7_inference_demonstrations.png').exists())]
stage7_validation_summary=pd.DataFrame(stage7_checks,columns=['check','passed']); stage7_validation_summary.to_csv('/kaggle/working/stage7_inference_validation_summary.csv',index=False); display(stage7_validation_summary); print(f"PASS checks: {int(stage7_validation_summary.passed.sum())} / {len(stage7_validation_summary)}"); assert stage7_validation_summary.passed.all()

## 7.5 Final Notebook Validation

This final audit checks the frozen Stage 1–4 results, executed Stage 5–7 work, and regenerated evidence. It does not generate a report or submission package.

In [ ]:
final_notebook_checks = [
    ('Stage 1 validation complete: 13/13 PASS', len(validation_summary) == 13 and bool(validation_summary['passed'].all())),
    ('Stage 1 PK audit complete and passed: 14/14', len(pk_audit) == 14 and bool(pk_audit['passed'].all())),
    ('Stage 1 FK unmatched count is zero', int(fk_audit['unmatched_rows'].sum()) == 0),
    ('Stage 2 validation complete: 24/24 PASS', len(stage2_validation_summary) == 24 and bool(stage2_validation_summary['passed'].all())),
    ('Stage 3 validation complete: 77/77 PASS', len(stage3_validation_summary) == 77 and bool(stage3_validation_summary['passed'].all())),
    ('Stage 3 modeling rows = 26,668', len(modeling_table) == 26668),
    ('Stage 3 selected predictors = 31', len(selected_predictors) == 31),
    ('Stage 3 transformed predictors = 306', len(transformed_feature_names) == 306),
    ('Stage 3 chronological split sizes match', [len(train_df), len(val_df), len(test_df)] == [23689, 1620, 1359]),
    ('Stage 4 validation complete: 60/60 PASS', len(stage4_validation_summary) == 60 and bool(stage4_validation_summary['passed'].all())),
    ('Stage 4 primary model is the frozen Shallow Neural Network', primary_model_name == 'Shallow Neural Network' and STAGE4_SELECTION_FROZEN),
    ('frozen primary threshold is in (0, 1)', 0 < PRIMARY_MODEL_THRESHOLD < 1),
    ('primary threshold matches Stage 4 selection', np.isclose(PRIMARY_MODEL_THRESHOLD, selected_thresholds[primary_model_name])),
    ('Stage 5 uses the same frozen threshold', np.allclose(stage5_ablation['frozen_threshold'], PRIMARY_MODEL_THRESHOLD)),
    ('Stage 6 uses the same frozen primary model', primary_model_name == 'Shallow Neural Network' and STAGE4_SELECTION_FROZEN),
    ('Stage 7 inference outputs use the frozen threshold', np.isclose(stage7_complete_result['threshold'], PRIMARY_MODEL_THRESHOLD) and np.isclose(stage7_partial_result['threshold'], PRIMARY_MODEL_THRESHOLD)),
    ('Stage 5 validation passes', bool(stage5_validation_summary['passed'].all())),
    ('Stage 5 all six feature groups completed', len(stage5_ablation) == 6 and set(stage5_ablation['removed_group']) == set(stage5_group_cols)),
    ('Stage 5 both preprocessing experiments completed', len(stage5_preprocessing_order_1) == 2 and len(stage5_preprocessing_order_2) == 2),
    ('Stage 5 did not use test data', STAGE5_TEST_USED is False),
    ('Stage 5 evidence regenerated', all((STAGE5_EVIDENCE_DIR / n).exists() for n in ['stage5_feature_group_ablation.csv','stage5_feature_group_ablation_pr_auc.png','stage5_preprocessing_order_experiment_1.csv','stage5_preprocessing_order_experiment_2.csv','stage5_validation_summary.csv'])),
    ('Stage 6 validation passes', bool(stage6_validation_summary['passed'].all())),
    ('Stage 6 permutation importance covers 31 raw predictors', len(stage6_permutation_importance) == 31),
    ('Stage 6 SHAP explanation completed', bool(stage6_shap_method)),
    ('Stage 6 LIME explanation completed', len(stage6_lime_table) > 0),
    ('Stage 6 non-causality disclaimer is present', 'do not establish causality' in stage6_xai_interpretation.lower() or 'do not establish that any feature causes' in stage6_xai_interpretation.lower() or 'do not establish causality' in stage6_xai_interpretation.lower()),
    ('Stage 6 did not use test data', STAGE6_TEST_USED is False),
    ('Stage 6 evidence regenerated', all((STAGE6_EVIDENCE_DIR / n).exists() for n in ['stage6_permutation_importance.png','stage6_shap_beeswarm.png','stage6_lime_local.png'])),
    ('Stage 7 inference function is callable', callable(predict_podium_companion)),
    ('Stage 7 complete demo probability is valid', 0 <= stage7_complete_result['probability'] <= 1),
    ('Stage 7 partial demo probability is valid and has four missing predictors', 0 <= stage7_partial_result['probability'] <= 1 and stage7_partial_result['missing_predictors'] >= 4),
    ('Stage 7 validation passes', bool(stage7_validation_summary['passed'].all())),
    ('Stage 7 evidence regenerated', all(Path('/kaggle/working/' + n).exists() for n in ['stage7_inference_complete_record.csv','stage7_inference_partial_record.csv','stage7_inference_demonstrations.png','stage7_inference_validation_summary.csv'])),
]
final_notebook_validation_summary = pd.DataFrame(final_notebook_checks, columns=['check', 'passed'])
final_notebook_validation_summary.to_csv('/kaggle/working/final_notebook_validation_summary.csv', index=False)
display(final_notebook_validation_summary)
print(f'FINAL NOTEBOOK PASS CHECKS: {int(final_notebook_validation_summary.passed.sum())} / {len(final_notebook_validation_summary)}')
assert final_notebook_validation_summary['passed'].all(), 'Final notebook validation failed.'